In [ ]:
# In[1]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Load files
df_metric = pd.read_csv('metric.csv')
df_trace = pd.read_csv('trace.csv')
df_log = pd.read_csv('log.csv')
df_error = pd.read_csv('error_logs.csv')

# Helper to convert timestamps and get min/max safely
def ts_summary(df):
    total = int(len(df))
    if total == 0:
        return {"count": 0, "min_ts": None, "max_ts": None, "min_iso": None, "max_iso": None}
    # ensure timestamp is integer
    df['timestamp'] = df['timestamp'].astype('int64')
    min_ts = int(df['timestamp'].min())
    max_ts = int(df['timestamp'].max())
    # create tz-aware datetimes in Asia/Shanghai
    dt_series = pd.to_datetime(df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    min_iso = dt_series.iloc[df['timestamp'].argmin()].isoformat()
    max_iso = dt_series.iloc[df['timestamp'].argmax()].isoformat()
    return {"count": total, "min_ts": min_ts, "max_ts": max_ts, "min_iso": min_iso, "max_iso": max_iso}

# Helper to get top N cmdb_id counts as list of dicts
def top_cmdb_counts(df, n=20):
    if len(df) == 0:
        return []
    vc = df['cmdb_id'].value_counts().head(n)
    return [{"cmdb_id": idx, "count": int(cnt)} for idx, cnt in vc.items()]

# Helper to get top N name counts and distinct count
def top_name_counts(df, colname, n=20):
    if len(df) == 0 or colname not in df.columns:
        return {"top": [], "distinct_count": 0}
    vc = df[colname].value_counts().head(n)
    top = [{"name": idx, "count": int(cnt)} for idx, cnt in vc.items()]
    distinct = int(df[colname].nunique(dropna=True))
    return {"top": top, "distinct_count": distinct}

# Summaries
metric_summary = ts_summary(df_metric)
metric_cmdb_top20 = top_cmdb_counts(df_metric, 20)
metric_kpi_summary = top_name_counts(df_metric, 'kpi_name', 20)

trace_summary = ts_summary(df_trace)
trace_cmdb_top20 = top_cmdb_counts(df_trace, 20)
trace_name_summary = top_name_counts(df_trace, 'trace_name', 20)

log_summary = ts_summary(df_log)
log_cmdb_top20 = top_cmdb_counts(df_log, 20)
log_name_summary = top_name_counts(df_log, 'log_name', 20)

# error_logs: top 20 distinct message samples (first-appearing distinct messages) and their counts
def error_messages_sample(df, n=20):
    total = int(len(df))
    if total == 0:
        return {"count": 0, "distinct_messages": 0, "samples": []}
    df['timestamp'] = df['timestamp'].astype('int64')
    # order by time ascending to get first appearance
    ordered = df.sort_values('timestamp', ascending=True)
    # first N distinct messages by appearance
    first_distinct = ordered.drop_duplicates(subset=['message'], keep='first').head(n)
    samples = []
    # counts for those messages
    counts = df['message'].value_counts()
    for msg in first_distinct['message'].tolist():
        samples.append({"message_sample": msg, "count": int(counts.get(msg, 0))})
    return {"count": total, "distinct_messages": int(df['message'].nunique(dropna=True)), "samples": samples}

error_summary = ts_summary(df_error)
error_cmdb_top20 = top_cmdb_counts(df_error, 20)
error_messages = error_messages_sample(df_error, 20)

# Assemble compact JSON-serializable result
result = {
    "metric.csv": {
        "rows": metric_summary["count"],
        "min_timestamp": metric_summary["min_ts"],
        "max_timestamp": metric_summary["max_ts"],
        "min_iso_ASIA_SHANGHAI": metric_summary["min_iso"],
        "max_iso_ASIA_SHANGHAI": metric_summary["max_iso"],
        "top_20_cmdb_id": metric_cmdb_top20,
        "kpi_name": {
            "top_20": metric_kpi_summary["top"],
            "distinct_count": metric_kpi_summary["distinct_count"]
        }
    },
    "trace.csv": {
        "rows": trace_summary["count"],
        "min_timestamp": trace_summary["min_ts"],
        "max_timestamp": trace_summary["max_ts"],
        "min_iso_ASIA_SHANGHAI": trace_summary["min_iso"],
        "max_iso_ASIA_SHANGHAI": trace_summary["max_iso"],
        "top_20_cmdb_id": trace_cmdb_top20,
        "trace_name": {
            "top_20": trace_name_summary["top"],
            "distinct_count": trace_name_summary["distinct_count"]
        }
    },
    "log.csv": {
        "rows": log_summary["count"],
        "min_timestamp": log_summary["min_ts"],
        "max_timestamp": log_summary["max_ts"],
        "min_iso_ASIA_SHANGHAI": log_summary["min_iso"],
        "max_iso_ASIA_SHANGHAI": log_summary["max_iso"],
        "top_20_cmdb_id": log_cmdb_top20,
        "log_name": {
            "top_20": log_name_summary["top"],
            "distinct_count": log_name_summary["distinct_count"]
        }
    },
    "error_logs.csv": {
        "rows": error_summary["count"],
        "min_timestamp": error_summary["min_ts"],
        "max_timestamp": error_summary["max_ts"],
        "min_iso_ASIA_SHANGHAI": error_summary["min_iso"],
        "max_iso_ASIA_SHANGHAI": error_summary["max_iso"],
        "top_20_cmdb_id": error_cmdb_top20,
        "messages": {
            "distinct_count": error_messages["distinct_messages"],
            "first_20_distinct_samples_with_counts": error_messages["samples"]
        }
    }
}

result

```
Out[1]:
```
```python
# Summarize the previously computed `result` dictionary in plain English.
# The `result` variable is expected to exist in the current IPython state.
res = result

summary = (
    f"Summary of telemetry files:\n\n"
    f"metric.csv: {res['metric.csv']['rows']:,} rows from {res['metric.csv']['min_iso_ASIA_SHANGHAI']} "
    f"to {res['metric.csv']['max_iso_ASIA_SHANGHAI']}. "
    f"Top cmdb_ids: "
    f"{', '.join([f\"{c['cmdb_id']} ({c['count']:,})\" for c in res['metric.csv']['top_20_cmdb_id'][:3]])} "
    f"(showing top 3 of 20). Total distinct kpi_name: {res['metric.csv']['kpi_name']['distinct_count']}.\n\n"
    f"trace.csv: {res['trace.csv']['rows']:,} rows from {res['trace.csv']['min_iso_ASIA_SHANGHAI']} "
    f"to {res['trace.csv']['max_iso_ASIA_SHANGHAI']}. "
    f"Top cmdb_ids: "
    f"{', '.join([f\"{c['cmdb_id']} ({c['count']:,})\" for c in res['trace.csv']['top_20_cmdb_id'][:3]])} "
    f"(top 3 of 20). Total distinct trace_name: {res['trace.csv']['trace_name']['distinct_count']}.\n\n"
    f"log.csv: {res['log.csv']['rows']:,} rows from {res['log.csv']['min_iso_ASIA_SHANGHAI']} "
    f"to {res['log.csv']['max_iso_ASIA_SHANGHAI']}. Top cmdb_ids include several pods at 60 rows each (e.g., "
    f"{', '.join([c['cmdb_id'] for c in res['log.csv']['top_20_cmdb_id'][:3]])}). "
    f"Top log_name values: "
    f"{', '.join([f\"{n['name']} ({n['count']})\" for n in res['log.csv']['log_name']['top_20']])}. "
    f"Distinct log_name count: {res['log.csv']['log_name']['distinct_count']}.\n\n"
    f"error_logs.csv: {res['error_logs.csv']['rows']} rows from {res['error_logs.csv']['min_iso_ASIA_SHANGHAI']} "
    f"to {res['error_logs.csv']['max_iso_ASIA_SHANGHAI']}. Top affected cmdb_ids: "
    f"{', '.join([f\"{c['cmdb_id']} ({c['count']})\" for c in res['error_logs.csv']['top_20_cmdb_id']])}. "
    f"Distinct messages: {res['error_logs.csv']['messages']['distinct_count']}. "
    f"Most frequent message sample: "
    f"\"{res['error_logs.csv']['messages']['first_20_distinct_samples_with_counts'][0]['message_sample']}\" "
    f"appears {res['error_logs.csv']['messages']['first_20_distinct_samples_with_counts'][0]['count']} times. "
    f"The remaining message samples (up to 7 shown) each appear once.\n\n"
    f"Key takeaway: metric data is large (321k+ rows) and spans 1 hour; trace and log data cover ~30 minutes. "
    f"Errors are concentrated in the frontend pods (frontend-0 and frontend-1 mainly), with a recurring "
    f"'severity: error, message: request error' message (7 occurrences) and several distinct HTTP 500 access log entries."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

{'metric.csv': {'rows': 321596, 'min_timestamp': 1647801000, 'max_timestamp': 1647804600, 'min_iso_ASIA_SHANGHAI': '2022-03-21T02:30:00+08:00', 'max_iso_ASIA_SHANGHAI': '2022-03-21T03:30:00+08:00', 'top_20_cmdb_id': [{'cmdb_id': 'adservice', 'count': 20801}, {'cmdb_id': 'adservice2', 'count': 20313}, {'cmdb_id': 'frontend2-0', 'count': 11956}, {'cmdb_id': 'frontend-2', 'count': 11468}, {'cmdb_id': 'frontend-1', 'count': 11468}, {'cmdb_id': 'frontend-0', 'count': 11468}, {'cmdb_id': 'checkoutservice-1', 'count': 8784}, {'cmdb_id': 'checkoutservice2-0', 'count': 8784}, {'cmdb_id': 'checkoutservice-2', 'count': 8540}, {'cmdb_id': 'checkoutservice-0', 'count': 8540}, {'cmdb_id': 'productcatalogservice-1', 'count': 7320}, {'cmdb_id': 'productcatalogservice-0', 'count': 7320}, {'cmdb_id': 'productcatalogservice-2', 'count': 7076}, {'cmdb_id': 'productcatalogservice2-0', 'count': 6344}, {'cmdb_id': 'recommendationservice2-0', 'count': 6222}, {'cmdb_id': 'adservice2-0', 'count': 6102}, {'cmdb_id': 'cartservice2-0', 'count': 6100}, {'cmdb_id': 'cartservice-2', 'count': 6100}, {'cmdb_id': 'cartservice-0', 'count': 6100}, {'cmdb_id': 'recommendationservice-2', 'count': 5978}], 'kpi_name': {'top_20': [{'name': 'container.node-6.container_network_transmit_errors.eth0', 'count': 2196}, {'name': 'container.node-6.container_network_transmit_packets_dropped.eth0', 'count': 2196}, {'name': 'container.node-6.container_network_transmit_packets.eth0', 'count': 2196}, {'name': 'container.node-6.container_network_transmit_MB.eth0', 'count': 2196}, {'name': 'container.node-6.container_network_receive_packets_dropped.eth0', 'count': 2196}, {'name': 'container.node-6.container_network_receive_errors.eth0', 'count': 2196}, {'name': 'container.node-6.container_network_receive_MB.eth0', 'count': 2196}, {'name': 'container.node-6.container_network_receive_packets.eth0', 'count': 2196}, {'name': 'container.node-6.container_start_time_seconds', 'count': 2074}, {'name': 'container.node-6.container_tasks_state.iowaiting', 'count': 2074}, {'name': 'container.node-6.container_spec_memory_reservation_limit_MB', 'count': 2074}, {'name': 'container.node-6.container_tasks_state.running', 'count': 2074}, {'name': 'container.node-6.container_tasks_state.sleeping', 'count': 2074}, {'name': 'container.node-6.container_memory_cache', 'count': 2074}, {'name': 'container.node-6.container_memory_max_usage_MB', 'count': 2074}, {'name': 'container.node-6.container_spec_memory_swap_limit_MB', 'count': 2074}, {'name': 'container.node-6.container_fs_reads_merged./dev/vda1', 'count': 2074}, {'name': 'container.node-6.container_fs_reads./dev/vda1', 'count': 2074}, {'name': 'container.node-6.container_fs_read_seconds./dev/vda1', 'count': 2074}, {'name': 'container.node-6.container_cpu_user_seconds', 'count': 2074}], 'distinct_count': 1388}}, 'trace.csv': {'rows': 28764, 'min_timestamp': 1647802800, 'max_timestamp': 1647804540, 'min_iso_ASIA_SHANGHAI': '2022-03-21T03:00:00+08:00', 'max_iso_ASIA_SHANGHAI': '2022-03-21T03:29:00+08:00', 'top_20_cmdb_id': [{'cmdb_id': 'frontend-1', 'count': 2552}, {'cmdb_id': 'frontend-0', 'count': 2544}, {'cmdb_id': 'frontend-2', 'count': 2348}, {'cmdb_id': 'checkoutservice-0', 'count': 1680}, {'cmdb_id': 'checkoutservice-2', 'count': 1540}, {'cmdb_id': 'checkoutservice-1', 'count': 1260}, {'cmdb_id': 'productcatalogservice-0', 'count': 1064}, {'cmdb_id': 'frontend2-0', 'count': 1056}, {'cmdb_id': 'productcatalogservice-1', 'count': 1052}, {'cmdb_id': 'productcatalogservice-2', 'count': 1052}, {'cmdb_id': 'recommendationservice-1', 'count': 840}, {'cmdb_id': 'recommendationservice-2', 'count': 840}, {'cmdb_id': 'checkoutservice2-0', 'count': 840}, {'cmdb_id': 'recommendationservice-0', 'count': 840}, {'cmdb_id': 'cartservice-0', 'count': 712}, {'cmdb_id': 'cartservice-1', 'count': 708}, {'cmdb_id': 'cartservice-2', 'count': 688}, {'cmdb_id': 'currencyservice-2', 'count': 624}, {'cmdb_id': 'currencyservice-0', 'count': 620}, {'cmdb_id': 'currencyservice-1', 'count': 620}], 'trace_name': {'top_20': [{'name': 'trace.from_frontend-1.row_count', 'count': 578}, {'name': 'trace.from_frontend-1.error_rate', 'count': 578}, {'name': 'trace.from_frontend-1.duration_p95', 'count': 578}, {'name': 'trace.from_frontend-1.duration_mean', 'count': 578}, {'name': 'trace.from_frontend-0.row_count', 'count': 576}, {'name': 'trace.from_frontend-0.error_rate', 'count': 576}, {'name': 'trace.from_frontend-0.duration_p95', 'count': 576}, {'name': 'trace.from_frontend-0.duration_mean', 'count': 576}, {'name': 'trace.from_frontend-2.duration_mean', 'count': 527}, {'name': 'trace.from_frontend-2.duration_p95', 'count': 527}, {'name': 'trace.from_frontend-2.error_rate', 'count': 527}, {'name': 'trace.from_frontend-2.row_count', 'count': 527}, {'name': 'trace.self.row_count', 'count': 456}, {'name': 'trace.self.error_rate', 'count': 456}, {'name': 'trace.self.duration_p95', 'count': 456}, {'name': 'trace.self.duration_mean', 'count': 456}, {'name': 'trace.from_checkoutservice-0.row_count', 'count': 327}, {'name': 'trace.from_checkoutservice-0.error_rate', 'count': 327}, {'name': 'trace.from_checkoutservice-0.duration_p95', 'count': 327}, {'name': 'trace.from_checkoutservice-0.duration_mean', 'count': 327}], 'distinct_count': 200}}, 'log.csv': {'rows': 1690, 'min_timestamp': 1647802800, 'max_timestamp': 1647804540, 'min_iso_ASIA_SHANGHAI': '2022-03-21T03:00:00+08:00', 'max_iso_ASIA_SHANGHAI': '2022-03-21T03:29:00+08:00', 'top_20_cmdb_id': [{'cmdb_id': 'adservice-0', 'count': 60}, {'cmdb_id': 'adservice-1', 'count': 60}, {'cmdb_id': 'adservice-2', 'count': 60}, {'cmdb_id': 'cartservice-0', 'count': 60}, {'cmdb_id': 'cartservice-1', 'count': 60}, {'cmdb_id': 'cartservice-2', 'count': 60}, {'cmdb_id': 'checkoutservice-0', 'count': 60}, {'cmdb_id': 'currencyservice-0', 'count': 60}, {'cmdb_id': 'currencyservice-1', 'count': 60}, {'cmdb_id': 'currencyservice-2', 'count': 60}, {'cmdb_id': 'frontend-0', 'count': 60}, {'cmdb_id': 'productcatalogservice-0', 'count': 60}, {'cmdb_id': 'productcatalogservice-1', 'count': 60}, {'cmdb_id': 'frontend-2', 'count': 60}, {'cmdb_id': 'frontend-1', 'count': 60}, {'cmdb_id': 'shippingservice-1', 'count': 60}, {'cmdb_id': 'shippingservice-2', 'count': 60}, {'cmdb_id': 'shippingservice-0', 'count': 60}, {'cmdb_id': 'recommendationservice-0', 'count': 60}, {'cmdb_id': 'productcatalogservice-2', 'count': 60}], 'log_name': {'top_20': [{'name': 'log.error_count', 'count': 845}, {'name': 'log.row_count', 'count': 845}], 'distinct_count': 2}}, 'error_logs.csv': {'rows': 14, 'min_timestamp': 1647803481, 'max_timestamp': 1647803567, 'min_iso_ASIA_SHANGHAI': '2022-03-21T03:11:21+08:00', 'max_iso_ASIA_SHANGHAI': '2022-03-21T03:12:47+08:00', 'top_20_cmdb_id': [{'cmdb_id': 'frontend-0', 'count': 6}, {'cmdb_id': 'frontend-1', 'count': 6}, {'cmdb_id': 'frontend-2', 'count': 2}], 'messages': {'distinct_count': 8, 'first_20_distinct_samples_with_counts': [{'message_sample': 'severity: error, message: request error', 'count': 7}, {'message_sample': '"POST /cart/checkout HTTP/1.1" 500 - via_upstream - "-" 256 4041 30635 30635 "-" "k6/0.26.2 (https://k6.io/)" "e5caff04-3fef-965b-a927-24758a19501f" "frontend.ts:80" "172.20.3.12:8080" inbound|8080|| 127.0.0.6:38397 172.20.3.12:8080 172.20.3.63:35070 - default', 'count': 1}, {'message_sample': '"POST /cart/checkout HTTP/1.1" 500 - via_upstream - "-" 256 4041 36618 36618 "-" "k6/0.26.2 (https://k6.io/)" "77739f62-4151-95fc-9548-2caae0da518a" "frontend.ts:80" "172.20.2.71:8080" inbound|8080|| 127.0.0.6:57073 172.20.2.71:8080 172.20.3.63:35086 - default', 'count': 1}, {'message_sample': '"POST /cart/checkout HTTP/1.1" 500 - via_upstream - "-" 256 4152 26631 26631 "-" "k6/0.26.2 (https://k6.io/)" "6723acde-0544-95a9-a14f-8078dde533db" "frontend.ts:80" "172.20.3.12:8080" inbound|8080|| 127.0.0.6:36111 172.20.3.12:8080 172.20.3.63:35044 - default', 'count': 1}, {'message_sample': '"POST /cart/checkout HTTP/1.1" 500 - via_upstream - "-" 256 4152 20427 20426 "-" "k6/0.26.2 (https://k6.io/)" "16b1df16-7abc-9501-8a60-3004a839ac09" "frontend.ts:80" "172.20.2.68:8080" inbound|8080|| 127.0.0.6:46833 172.20.2.68:8080 172.20.3.63:35036 - default', 'count': 1}, {'message_sample': '"POST /cart/checkout HTTP/1.1" 500 - via_upstream - "-" 256 4041 25631 25630 "-" "k6/0.26.2 (https://k6.io/)" "98416ede-3d96-97ba-a42b-f6038b1a9752" "frontend.ts:80" "172.20.2.68:8080" inbound|8080|| 127.0.0.6:42511 172.20.2.68:8080 172.20.3.63:35062 - default', 'count': 1}, {'message_sample': '"POST /cart/checkout HTTP/1.1" 500 - via_upstream - "-" 256 4041 58217 58217 "-" "k6/0.26.2 (https://k6.io/)" "7fa2a36a-1f5c-98d1-8e08-7f217942c1f3" "frontend.ts:80" "172.20.2.68:8080" inbound|8080|| 127.0.0.6:49019 172.20.2.68:8080 172.20.3.63:35096 - default', 'count': 1}, {'message_sample': '"POST /cart/checkout HTTP/1.1" 500 - via_upstream - "-" 256 4152 20208 20207 "-" "k6/0.26.2 (https://k6.io/)" "c1f3614d-441e-915c-9a51-0cd6538a2e02" "frontend.ts:80" "172.20.3.12:8080" inbound|8080|| 127.0.0.6:38397 172.20.3.12:8080 172.20.3.63:35100 - default', 'count': 1}]}}}```
```

In [ ]:
# In[2]:

import pandas as pd
import pytz

# Reuse existing df_metric and tz if available; otherwise load/define
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv')
try:
    tz
except NameError:
    tz = pytz.timezone('Asia/Shanghai')

# Ensure timestamp is integer
df_metric['timestamp'] = df_metric['timestamp'].astype('int64')

# Target components
targets = [
    "frontend-0","frontend-1","frontend-2",
    "checkoutservice-0","checkoutservice-1","checkoutservice-2",
    "cartservice-0","cartservice-1","cartservice-2"
]

# Filter rows
filtered = df_metric[df_metric['cmdb_id'].isin(targets)].copy()

# 1) total row count and per-cmdb_id counts (one row per component)
total_count = int(filtered.shape[0])
per_cmdb_counts_series = filtered['cmdb_id'].value_counts().reindex(targets, fill_value=0)
per_cmdb_counts = [{"cmdb_id": cid, "count": int(per_cmdb_counts_series[cid])} for cid in targets]

# 2) global min and max timestamps for filtered set (and ISO in Asia/Shanghai)
if total_count == 0:
    min_ts = None
    max_ts = None
    min_iso = None
    max_iso = None
else:
    min_ts = int(filtered['timestamp'].min())
    max_ts = int(filtered['timestamp'].max())
    min_iso = pd.to_datetime(min_ts, unit='s', utc=True).tz_convert(tz).isoformat()
    max_iso = pd.to_datetime(max_ts, unit='s', utc=True).tz_convert(tz).isoformat()

# 3) top 50 kpi_name across filtered set with counts
top50_kpi_series = filtered['kpi_name'].value_counts().head(50)
top_50_kpi = [{"kpi_name": name, "count": int(cnt)} for name, cnt in top50_kpi_series.items()]

# 4) for each cmdb_id, top 10 kpi_name values with counts
# Use groupby value_counts for efficiency
top_kpi_per_cmdb = {}
if total_count > 0:
    grp = filtered.groupby('cmdb_id')['kpi_name'].value_counts()
    for cid in targets:
        if cid in grp.index.get_level_values(0):
            s = grp[cid].head(10)
            top_kpi_per_cmdb[cid] = [{"kpi_name": name, "count": int(cnt)} for name, cnt in s.items()]
        else:
            top_kpi_per_cmdb[cid] = []

# Assemble compact JSON-serializable summary
summary = {
    "filtered_components": targets,
    "total_rows_filtered": total_count,
    "per_cmdb_counts": per_cmdb_counts,
    "global_min_timestamp": min_ts,
    "global_max_timestamp": max_ts,
    "global_min_iso_ASIA_SHANGHAI": min_iso,
    "global_max_iso_ASIA_SHANGHAI": max_iso,
    "top_50_kpi_names": top_50_kpi,
    "top_10_kpi_per_cmdb": top_kpi_per_cmdb
}

summary

```
Out[2]:
```
```python
# Summarize the filtered metric.csv results (stored in `summary`) in plain English.
# Reuse the existing `summary` variable from the previous step and present a compact text summary.
final_summary = (
    f"Filtered metric.csv summary (components: frontend-0/1/2, checkoutservice-0/1/2, cartservice-0/1/2):\n\n"
    f"1) Total rows in filtered set: {summary['total_rows_filtered']:,}.\n"
    f"   Per-component counts: "
    f"{', '.join([f\"{c['cmdb_id']}={c['count']:,}\" for c in summary['per_cmdb_counts']])}.\n\n"
    f"2) Time range for filtered data: {summary['global_min_iso_ASIA_SHANGHAI']} to {summary['global_max_iso_ASIA_SHANGHAI']}.\n\n"
    f"3) Top KPI names across the filtered set (top 3 of up to 50 shown):\n"
    f"   - {summary['top_50_kpi_names'][0]['kpi_name']} ({summary['top_50_kpi_names'][0]['count']})\n"
    f"   - {summary['top_50_kpi_names'][1]['kpi_name']} ({summary['top_50_kpi_names'][1]['count']})\n"
    f"   - {summary['top_50_kpi_names'][2]['kpi_name']} ({summary['top_50_kpi_names'][2]['count']})\n"
    f"   (A total of {len(summary['top_50_kpi_names'])} KPI entries are listed in the top-50 slice.)\n\n"
    f"4) Per-component top KPIs (top 1-3 examples shown for each component):\n"
    + "\n".join(
        f"   {cid}: {', '.join([f\"{item['kpi_name']} ({item['count']})\" for item in summary['top_10_kpi_per_cmdb'][cid][:3]])}"
        for cid in summary['top_10_kpi_per_cmdb']
    )
)

final_summary
```

The original code execution output of IPython Kernel is also provided below for reference:

{'filtered_components': ['frontend-0', 'frontend-1', 'frontend-2', 'checkoutservice-0', 'checkoutservice-1', 'checkoutservice-2', 'cartservice-0', 'cartservice-1', 'cartservice-2'], 'total_rows_filtered': 78324, 'per_cmdb_counts': [{'cmdb_id': 'frontend-0', 'count': 11468}, {'cmdb_id': 'frontend-1', 'count': 11468}, {'cmdb_id': 'frontend-2', 'count': 11468}, {'cmdb_id': 'checkoutservice-0', 'count': 8540}, {'cmdb_id': 'checkoutservice-1', 'count': 8784}, {'cmdb_id': 'checkoutservice-2', 'count': 8540}, {'cmdb_id': 'cartservice-0', 'count': 6100}, {'cmdb_id': 'cartservice-1', 'count': 5856}, {'cmdb_id': 'cartservice-2', 'count': 6100}], 'global_min_timestamp': 1647801000, 'global_max_timestamp': 1647804600, 'global_min_iso_ASIA_SHANGHAI': '2022-03-21T02:30:00+08:00', 'global_max_iso_ASIA_SHANGHAI': '2022-03-21T03:30:00+08:00', 'top_50_kpi_names': [{'kpi_name': 'container.node-6.container_fs_writes./dev/vda1', 'count': 366}, {'kpi_name': 'container.node-6.container_fs_write_seconds./dev/vda1', 'count': 366}, {'kpi_name': 'container.node-6.container_fs_usage_MB./dev/vda1', 'count': 366}, {'kpi_name': 'container.node-6.container_fs_sector_writes./dev/vda1', 'count': 366}, {'kpi_name': 'container.node-6.container_fs_sector_reads./dev/vda1', 'count': 366}, {'kpi_name': 'container.node-6.container_fs_reads_merged./dev/vda1', 'count': 366}, {'kpi_name': 'container.node-6.container_fs_reads_MB./dev/vda', 'count': 366}, {'kpi_name': 'container.node-6.container_fs_reads./dev/vda1', 'count': 366}, {'kpi_name': 'container.node-6.container_network_transmit_packets_dropped.eth0', 'count': 366}, {'kpi_name': 'container.node-6.container_cpu_user_seconds', 'count': 366}, {'kpi_name': 'container.node-6.container_threads', 'count': 366}, {'kpi_name': 'container.node-6.container_spec_memory_swap_limit_MB', 'count': 366}, {'kpi_name': 'container.node-6.container_spec_memory_reservation_limit_MB', 'count': 366}, {'kpi_name': 'container.node-6.container_spec_memory_limit_MB', 'count': 366}, {'kpi_name': 'container.node-6.container_spec_cpu_shares', 'count': 366}, {'kpi_name': 'container.node-6.container_spec_cpu_quota', 'count': 366}, {'kpi_name': 'container.node-6.container_spec_cpu_period', 'count': 366}, {'kpi_name': 'container.node-6.container_tasks_state.iowaiting', 'count': 366}, {'kpi_name': 'container.node-6.container_sockets', 'count': 366}, {'kpi_name': 'container.node-6.container_ulimits_soft.max_open_files', 'count': 366}, {'kpi_name': 'container.node-6.container_threads_max', 'count': 366}, {'kpi_name': 'container.node-6.container_start_time_seconds', 'count': 366}, {'kpi_name': 'container.node-6.container_network_transmit_packets.eth0', 'count': 366}, {'kpi_name': 'container.node-6.container_network_transmit_errors.eth0', 'count': 366}, {'kpi_name': 'container.node-6.container_network_transmit_MB.eth0', 'count': 366}, {'kpi_name': 'container.node-6.container_network_receive_packets_dropped.eth0', 'count': 366}, {'kpi_name': 'container.node-6.container_network_receive_packets.eth0', 'count': 366}, {'kpi_name': 'container.node-6.container_network_receive_errors.eth0', 'count': 366}, {'kpi_name': 'container.node-6.container_network_receive_MB.eth0', 'count': 366}, {'kpi_name': 'container.node-6.container_memory_working_set_MB', 'count': 366}, {'kpi_name': 'container.node-6.container_memory_usage_MB', 'count': 366}, {'kpi_name': 'container.node-6.container_memory_swap', 'count': 366}, {'kpi_name': 'container.node-6.container_fs_writes_MB./dev/vda', 'count': 366}, {'kpi_name': 'container.node-6.container_memory_max_usage_MB', 'count': 366}, {'kpi_name': 'container.node-6.container_memory_mapped_file', 'count': 366}, {'kpi_name': 'container.node-6.container_memory_failures.hierarchy.pgmajfault', 'count': 366}, {'kpi_name': 'container.node-6.container_memory_failures.hierarchy.pgfault', 'count': 366}, {'kpi_name': 'container.node-6.container_memory_failures.container.pgmajfault', 'count': 366}, {'kpi_name': 'container.node-6.container_memory_failures.container.pgfault', 'count': 366}, {'kpi_name': 'container.node-6.container_memory_failcnt', 'count': 366}, {'kpi_name': 'container.node-6.container_memory_cache', 'count': 366}, {'kpi_name': 'container.node-6.container_last_seen', 'count': 366}, {'kpi_name': 'container.node-6.container_fs_writes_merged./dev/vda1', 'count': 366}, {'kpi_name': 'container.node-6.container_memory_rss', 'count': 366}, {'kpi_name': 'container.node-6.container_fs_read_seconds./dev/vda1', 'count': 366}, {'kpi_name': 'container.node-6.container_cpu_cfs_periods', 'count': 366}, {'kpi_name': 'container.node-6.container_cpu_cfs_throttled_periods', 'count': 366}, {'kpi_name': 'container.node-6.container_cpu_cfs_throttled_seconds', 'count': 366}, {'kpi_name': 'container.node-6.container_cpu_load_average_10s', 'count': 366}, {'kpi_name': 'container.node-6.container_cpu_system_seconds', 'count': 366}], 'top_10_kpi_per_cmdb': {'frontend-0': [{'kpi_name': 'container.node-6.container_cpu_cfs_periods', 'count': 61}, {'kpi_name': 'container.node-6.container_cpu_cfs_throttled_periods', 'count': 61}, {'kpi_name': 'container.node-6.container_cpu_cfs_throttled_seconds', 'count': 61}, {'kpi_name': 'container.node-6.container_cpu_load_average_10s', 'count': 61}, {'kpi_name': 'container.node-6.container_cpu_system_seconds', 'count': 61}, {'kpi_name': 'container.node-6.container_cpu_usage_seconds', 'count': 61}, {'kpi_name': 'container.node-6.container_cpu_user_seconds', 'count': 61}, {'kpi_name': 'container.node-6.container_file_descriptors', 'count': 61}, {'kpi_name': 'container.node-6.container_fs_inodes./dev/vda1', 'count': 61}, {'kpi_name': 'container.node-6.container_fs_inodes_free./dev/vda1', 'count': 61}], 'frontend-1': [{'kpi_name': 'container.node-5.container_cpu_cfs_periods', 'count': 61}, {'kpi_name': 'container.node-5.container_cpu_cfs_throttled_periods', 'count': 61}, {'kpi_name': 'container.node-5.container_cpu_cfs_throttled_seconds', 'count': 61}, {'kpi_name': 'container.node-5.container_cpu_load_average_10s', 'count': 61}, {'kpi_name': 'container.node-5.container_cpu_system_seconds', 'count': 61}, {'kpi_name': 'container.node-5.container_cpu_usage_seconds', 'count': 61}, {'kpi_name': 'container.node-5.container_cpu_user_seconds', 'count': 61}, {'kpi_name': 'container.node-5.container_file_descriptors', 'count': 61}, {'kpi_name': 'container.node-5.container_fs_inodes./dev/vda1', 'count': 61}, {'kpi_name': 'container.node-5.container_fs_inodes_free./dev/vda1', 'count': 61}], 'frontend-2': [{'kpi_name': 'container.node-5.container_cpu_cfs_periods', 'count': 61}, {'kpi_name': 'container.node-5.container_cpu_cfs_throttled_periods', 'count': 61}, {'kpi_name': 'container.node-5.container_cpu_cfs_throttled_seconds', 'count': 61}, {'kpi_name': 'container.node-5.container_cpu_load_average_10s', 'count': 61}, {'kpi_name': 'container.node-5.container_cpu_system_seconds', 'count': 61}, {'kpi_name': 'container.node-5.container_cpu_usage_seconds', 'count': 61}, {'kpi_name': 'container.node-5.container_cpu_user_seconds', 'count': 61}, {'kpi_name': 'container.node-5.container_file_descriptors', 'count': 61}, {'kpi_name': 'container.node-5.container_fs_inodes./dev/vda1', 'count': 61}, {'kpi_name': 'container.node-5.container_fs_inodes_free./dev/vda1', 'count': 61}], 'checkoutservice-0': [{'kpi_name': 'container.node-6.container_cpu_cfs_periods', 'count': 61}, {'kpi_name': 'container.node-6.container_cpu_cfs_throttled_periods', 'count': 61}, {'kpi_name': 'container.node-6.container_cpu_cfs_throttled_seconds', 'count': 61}, {'kpi_name': 'container.node-6.container_cpu_load_average_10s', 'count': 61}, {'kpi_name': 'container.node-6.container_cpu_system_seconds', 'count': 61}, {'kpi_name': 'container.node-6.container_cpu_usage_seconds', 'count': 61}, {'kpi_name': 'container.node-6.container_cpu_user_seconds', 'count': 61}, {'kpi_name': 'container.node-6.container_file_descriptors', 'count': 61}, {'kpi_name': 'container.node-6.container_fs_inodes./dev/vda1', 'count': 61}, {'kpi_name': 'container.node-6.container_fs_inodes_free./dev/vda1', 'count': 61}], 'checkoutservice-1': [{'kpi_name': 'container.node-6.container_cpu_cfs_periods', 'count': 61}, {'kpi_name': 'container.node-6.container_cpu_cfs_throttled_periods', 'count': 61}, {'kpi_name': 'container.node-6.container_cpu_cfs_throttled_seconds', 'count': 61}, {'kpi_name': 'container.node-6.container_cpu_load_average_10s', 'count': 61}, {'kpi_name': 'container.node-6.container_cpu_system_seconds', 'count': 61}, {'kpi_name': 'container.node-6.container_cpu_usage_seconds', 'count': 61}, {'kpi_name': 'container.node-6.container_cpu_user_seconds', 'count': 61}, {'kpi_name': 'container.node-6.container_file_descriptors', 'count': 61}, {'kpi_name': 'container.node-6.container_fs_inodes./dev/vda1', 'count': 61}, {'kpi_name': 'container.node-6.container_fs_inodes_free./dev/vda1', 'count': 61}], 'checkoutservice-2': [{'kpi_name': 'container.node-5.container_cpu_cfs_periods', 'count': 61}, {'kpi_name': 'container.node-5.container_cpu_cfs_throttled_periods', 'count': 61}, {'kpi_name': 'container.node-5.container_cpu_cfs_throttled_seconds', 'count': 61}, {'kpi_name': 'container.node-5.container_cpu_load_average_10s', 'count': 61}, {'kpi_name': 'container.node-5.container_cpu_system_seconds', 'count': 61}, {'kpi_name': 'container.node-5.container_cpu_usage_seconds', 'count': 61}, {'kpi_name': 'container.node-5.container_cpu_user_seconds', 'count': 61}, {'kpi_name': 'container.node-5.container_file_descriptors', 'count': 61}, {'kpi_name': 'container.node-5.container_fs_inodes./dev/vda1', 'count': 61}, {'kpi_name': 'container.node-5.container_fs_inodes_free./dev/vda1', 'count': 61}], 'cartservice-0': [{'kpi_name': 'container.node-6.container_cpu_cfs_periods', 'count': 61}, {'kpi_name': 'container.node-6.container_cpu_cfs_throttled_periods', 'count': 61}, {'kpi_name': 'container.node-6.container_cpu_cfs_throttled_seconds', 'count': 61}, {'kpi_name': 'container.node-6.container_cpu_load_average_10s', 'count': 61}, {'kpi_name': 'container.node-6.container_cpu_system_seconds', 'count': 61}, {'kpi_name': 'container.node-6.container_cpu_usage_seconds', 'count': 61}, {'kpi_name': 'container.node-6.container_cpu_user_seconds', 'count': 61}, {'kpi_name': 'container.node-6.container_file_descriptors', 'count': 61}, {'kpi_name': 'container.node-6.container_fs_inodes./dev/vda1', 'count': 61}, {'kpi_name': 'container.node-6.container_fs_inodes_free./dev/vda1', 'count': 61}], 'cartservice-1': [{'kpi_name': 'container.node-6.container_cpu_cfs_periods', 'count': 61}, {'kpi_name': 'container.node-6.container_cpu_cfs_throttled_periods', 'count': 61}, {'kpi_name': 'container.node-6.container_cpu_cfs_throttled_seconds', 'count': 61}, {'kpi_name': 'container.node-6.container_cpu_load_average_10s', 'count': 61}, {'kpi_name': 'container.node-6.container_cpu_system_seconds', 'count': 61}, {'kpi_name': 'container.node-6.container_cpu_usage_seconds', 'count': 61}, {'kpi_name': 'container.node-6.container_cpu_user_seconds', 'count': 61}, {'kpi_name': 'container.node-6.container_file_descriptors', 'count': 61}, {'kpi_name': 'container.node-6.container_fs_inodes./dev/vda1', 'count': 61}, {'kpi_name': 'container.node-6.container_fs_inodes_free./dev/vda1', 'count': 61}], 'cartservice-2': [{'kpi_name': 'container.node-6.container_cpu_cfs_periods', 'count': 61}, {'kpi_name': 'container.node-6.container_cpu_cfs_throttled_periods', 'count': 61}, {'kpi_name': 'container.node-6.container_cpu_cfs_throttled_seconds', 'count': 61}, {'kpi_name': 'container.node-6.container_cpu_load_average_10s', 'count': 61}, {'kpi_name': 'container.node-6.container_cpu_system_seconds', 'count': 61}, {'kpi_name': 'container.node-6.container_cpu_usage_seconds', 'count': 61}, {'kpi_name': 'container.node-6.container_cpu_user_seconds', 'count': 61}, {'kpi_name': 'container.node-6.container_file_descriptors', 'count': 61}, {'kpi_name': 'container.node-6.container_fs_inodes./dev/vda1', 'count': 61}, {'kpi_name': 'container.node-6.container_fs_inodes_free./dev/vda1', 'count': 61}]}}```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np
import pytz

# Reuse existing df_metric if present; otherwise load
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv')

# Ensure types
df_metric['timestamp'] = df_metric['timestamp'].astype('int64')
df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')

tz = pytz.timezone('Asia/Shanghai')

# Target components (reuse from earlier)
targets = [
    "frontend-0","frontend-1","frontend-2",
    "checkoutservice-0","checkoutservice-1","checkoutservice-2",
    "cartservice-0","cartservice-1","cartservice-2"
]

# 1) Identify unique (cmdb_id, kpi_name) pairs present in the filtered set
filtered = df_metric[df_metric['cmdb_id'].isin(targets)]
pairs_df = filtered[['cmdb_id', 'kpi_name']].drop_duplicates().reset_index(drop=True)

# Compute global P95 for all (cmdb_id, kpi_name) groups across the entire metric.csv (rule 9)
p95_all = (
    df_metric
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)['value']
    .quantile(0.95)
    .rename(columns={'value': 'P95_value'})
)

# Restrict P95 to only pairs present in the filtered set
p95_pairs = pairs_df.merge(p95_all, on=['cmdb_id', 'kpi_name'], how='left')

# 2) Filter metric rows to the incident time window (inclusive)
start_ts = 1647802800
end_ts = 1647804540
window_mask = (df_metric['timestamp'] >= start_ts) & (df_metric['timestamp'] <= end_ts) & (df_metric['cmdb_id'].isin(targets))
df_window = df_metric[window_mask].copy()

# 3) For each pair, compute aggregates within the incident window
# Window counts and max
window_counts = (
    df_window
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)
    .agg(total_points_in_window=('value', 'size'),
         max_value_in_window=('value', 'max'))
)

# Anomaly computations: need P95 per pair joined to window rows to detect exceedances
df_window_with_p95 = df_window.merge(p95_pairs, on=['cmdb_id', 'kpi_name'], how='inner')

# Flag exceedances (value > P95). If P95 is NaN then no exceedance.
df_window_with_p95['is_exceed'] = df_window_with_p95['value'] > df_window_with_p95['P95_value']

# Aggregate exceedance counts and earliest anomalous timestamp per pair
anom_stats = (
    df_window_with_p95[df_window_with_p95['is_exceed']]
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)
    .agg(count_points_exceeding_P95=('is_exceed', 'sum'),
         earliest_anomalous_timestamp=('timestamp', 'min'))
)

# Merge everything: start from p95_pairs to include pairs with zero points in window
agg = (
    p95_pairs
    .merge(window_counts, on=['cmdb_id', 'kpi_name'], how='left')
    .merge(anom_stats, on=['cmdb_id', 'kpi_name'], how='left')
)

# Fill NaNs: total_points -> 0, counts -> 0, earliest timestamp -> None, max_value -> NaN (keep NaN if no data)
agg['total_points_in_window'] = agg['total_points_in_window'].fillna(0).astype(int)
agg['count_points_exceeding_P95'] = agg['count_points_exceeding_P95'].fillna(0).astype(int)
agg['earliest_anomalous_timestamp'] = agg['earliest_anomalous_timestamp'].where(agg['earliest_anomalous_timestamp'].notna(), None)
# max_value_in_window keep as float; if missing, remains NaN

# 4) Prepare final sorted result and limit to top 50 by count_points_exceeding_P95 desc
final_df = agg[[
    'cmdb_id', 'kpi_name', 'P95_value',
    'total_points_in_window', 'count_points_exceeding_P95',
    'earliest_anomalous_timestamp', 'max_value_in_window'
]].sort_values(by=['count_points_exceeding_P95', 'total_points_in_window'], ascending=[False, False])

# Ensure column exists for max_value_in_window (may be named differently if absent)
if 'max_value_in_window' not in final_df.columns and 'max_value_in_window' in agg.columns:
    final_df['max_value_in_window'] = agg['max_value_in_window']

# Limit to top 50
result_df = final_df.head(50).reset_index(drop=True)

# Compact display: show shape and head (top results). The variable `result_df` is the final output.
result_df.shape, result_df

```
Out[3]:
```
```python
# Summarize the computed anomaly aggregates for filtered components.
# Reuse existing variables from previous execution: result_df (top 50 rows) and final_df/agg if available.

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Prefer to use 'final_df' (full sorted results) and 'result_df' (top 50) if present.
try:
    total_pairs = final_df.shape[0]
except NameError:
    # fallback: try agg
    try:
        total_pairs = agg.shape[0]
    except NameError:
        total_pairs = None

try:
    top50 = result_df.copy()
except NameError:
    top50 = None

# Count how many pairs had any exceedances in the full set (if agg exists)
try:
    num_pairs_with_exceed = int((agg['count_points_exceeding_P95'] > 0).sum())
except Exception:
    num_pairs_with_exceed = None

# Build concise top-5 summary from the returned top50 slice
top_n = 5
top_entries = []
if top50 is not None and top50.shape[0] > 0:
    for i in range(min(top_n, top50.shape[0])):
        row = top50.iloc[i]
        et = row['earliest_anomalous_timestamp']
        if et is None or (pd.isna(et)):
            et_iso = None
        else:
            # ensure integer seconds
            et_iso = pd.to_datetime(int(et), unit='s', utc=True).tz_convert(tz).isoformat()
        top_entries.append({
            "cmdb_id": row['cmdb_id'],
            "kpi_name": row['kpi_name'],
            "P95_value": float(row['P95_value']) if not pd.isna(row['P95_value']) else None,
            "total_points_in_window": int(row['total_points_in_window']),
            "count_points_exceeding_P95": int(row['count_points_exceeding_P95']),
            "earliest_anomalous_timestamp_iso_ASIA_SHANGHAI": et_iso,
            "max_value_in_window": float(row['max_value_in_window']) if not pd.isna(row['max_value_in_window']) else None
        })

# Compose final plain-English summary string (compact)
summary = (
    f"Anomaly summary for filtered components within 2022-03-21 03:00:00+08 to 03:29:00+08:\n"
    f"- Total distinct (cmdb_id, kpi_name) pairs evaluated: {total_pairs if total_pairs is not None else 'unknown'}.\n"
    f"- Pairs with at least one value > global P95 during the window: {num_pairs_with_exceed if num_pairs_with_exceed is not None else 'unknown'}.\n"
    f"- Returned top {top50.shape[0] if top50 is not None else 0} pairs sorted by number of exceedances (count_points_exceeding_P95).\n\n"
    f"Top {min(top_n, top50.shape[0] if top50 is not None else 0)} pairs (cmdb_id, kpi_name) with sample details:\n"
)

for e in top_entries:
    summary += (
        f"  - {e['cmdb_id']} | {e['kpi_name']}\n"
        f"      P95: {e['P95_value']}, points_in_window: {e['total_points_in_window']}, "
        f"exceed_count: {e['count_points_exceeding_P95']}, earliest_exceed: {e['earliest_anomalous_timestamp_iso_ASIA_SHANGHAI']}, "
        f"max_in_window: {e['max_value_in_window']}\n"
    )

summary += "\nNote: P95 thresholds were computed globally using all available historical rows per (cmdb_id,kpi_name) before filtering to the incident window."

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

((50, 7),               cmdb_id                                           kpi_name     P95_value  total_points_in_window  count_points_exceeding_P95  earliest_anomalous_timestamp  max_value_in_window
0       cartservice-1         container.node-6.container_cpu_cfs_periods  1.640000e+02                      30                           3                  1.647803e+09         1.710000e+02
1       cartservice-2       container.node-6.container_cpu_usage_seconds  3.580729e-01                      30                           3                  1.647803e+09         3.885061e-01
2       cartservice-2        container.node-6.container_cpu_user_seconds  2.950000e-01                      30                           3                  1.647803e+09         3.350000e-01
3   checkoutservice-0        container.node-6.container_cpu_user_seconds  3.500000e-02                      30                           3                  1.647803e+09         4.000000e-02
4   checkoutservice-0  mesh.source.checkoutservice.jaeger-collector.i...  1.575000e-03                      30                           3                  1.647804e+09         9.650525e+00
..                ...                                                ...           ...                     ...                         ...                           ...                  ...
45      cartservice-2   container.node-6.container_memory_working_set_MB  1.311914e+02                      30                           2                  1.647804e+09         1.313184e+02
46      cartservice-2  mesh.destination.frontend.cartservice.istio_re...  3.147500e-02                      30                           2                  1.647803e+09         3.287500e-02
47  checkoutservice-0      container.node-6.container_cpu_system_seconds  2.500000e-02                      30                           2                  1.647803e+09         3.000000e-02
48  checkoutservice-0       container.node-6.container_cpu_usage_seconds  5.879057e-02                      30                           2                  1.647804e+09         6.589975e-02
49  checkoutservice-0               container.node-6.container_last_seen  1.647804e+09                      30                           2                  1.647804e+09         1.647805e+09

[50 rows x 7 columns])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np
import pytz

# Reuse existing variables if available
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv')
df_metric['timestamp'] = df_metric['timestamp'].astype('int64')
df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')

try:
    result_df  # top-50 pairs sorted by exceedances from previous step
except NameError:
    raise RuntimeError("Expected variable `result_df` from prior computation not found in state.")

# Incident window bounds (from prior steps)
start_ts = 1647802800
end_ts = 1647804540

# Use df_window if present; otherwise build it
try:
    df_window
except NameError:
    window_mask = (df_metric['timestamp'] >= start_ts) & (df_metric['timestamp'] <= end_ts)
    df_window = df_metric[window_mask].copy()

# Top 10 pairs from previous sorted results
top10_pairs = result_df[['cmdb_id', 'kpi_name', 'P95_value']].head(10).reset_index(drop=True)

runs = []
dropped_counts = []

for _, row in top10_pairs.iterrows():
    cid = row['cmdb_id']
    kpi = row['kpi_name']
    p95 = row['P95_value']
    # Extract window rows for this exact pair
    pr = df_window[(df_window['cmdb_id'] == cid) & (df_window['kpi_name'] == kpi)].sort_values('timestamp').copy()
    # If no rows in window, record zero drops and continue
    if pr.shape[0] == 0:
        dropped_counts.append({"cmdb_id": cid, "kpi_name": kpi, "single_point_exceedances_dropped": 0})
        continue
    # Determine exceedances using pre-computed global P95 (if NaN, no exceedances)
    if pd.isna(p95):
        exceeded = pr.iloc[0:0]  # empty
    else:
        exceeded = pr[pr['value'] > p95].copy()
    if exceeded.empty:
        dropped_counts.append({"cmdb_id": cid, "kpi_name": kpi, "single_point_exceedances_dropped": 0})
        continue
    # Identify consecutive runs: sort by timestamp, compute diffs
    exceeded = exceeded.sort_values('timestamp').reset_index(drop=True)
    exceeded['ts_diff'] = exceeded['timestamp'].diff().fillna(999999)  # large for first
    # New run when diff > 60
    exceeded['run_id'] = (exceeded['ts_diff'] > 60).cumsum()
    groups = exceeded.groupby('run_id')
    pair_runs = []
    single_point_drops = 0
    for gid, g in groups:
        run_len = len(g)
        run_start = int(g['timestamp'].min())
        run_end = int(g['timestamp'].max())
        run_max = float(g['value'].max())
        # Exclude runs of length 1 from primary output, count as dropped
        if run_len == 1:
            single_point_drops += 1
            continue
        # compute severity_ratio if possible
        if pd.isna(p95) or p95 == 0:
            severity = np.nan
        else:
            severity = float(run_max) / float(p95)
        pair_runs.append({
            "cmdb_id": cid,
            "kpi_name": kpi,
            "run_start_timestamp": run_start,
            "run_end_timestamp": run_end,
            "run_length_in_points": int(run_len),
            "run_max_value": run_max,
            "P95_value": float(p95) if not pd.isna(p95) else None,
            "severity_ratio": severity
        })
    # Limit to top 5 runs per pair by severity_ratio desc
    if pair_runs:
        pruns_df = pd.DataFrame(pair_runs)
        pruns_df = pruns_df.sort_values(by='severity_ratio', ascending=False, na_position='last').head(5)
        # extend to global runs list
        runs.extend(pruns_df.to_dict('records'))
    dropped_counts.append({"cmdb_id": cid, "kpi_name": kpi, "single_point_exceedances_dropped": int(single_point_drops)})

# Assemble final DataFrame of runs, sort by severity_ratio desc, limit to 50
runs_df = pd.DataFrame(runs)
if not runs_df.empty:
    runs_df = runs_df.sort_values(by='severity_ratio', ascending=False, na_position='last').reset_index(drop=True)
    runs_df = runs_df.head(50)

dropped_df = pd.DataFrame(dropped_counts)

# Compact output: `runs_df` (top runs) and `dropped_df` (per-pair dropped single-point counts)
runs_df, dropped_df

```
Out[4]:
```
```python
# Summarize anomalous runs detected for the top-10 (cmdb_id, kpi_name) pairs.
# Reuse `runs_df` and `dropped_df` produced in the previous step.

import pandas as pd
import pytz
tz = pytz.timezone('Asia/Shanghai')

try:
    runs_df
    dropped_df
except NameError:
    raise RuntimeError("Expected variables `runs_df` and `dropped_df` missing from state.")

# Compact counts
total_runs_returned = int(runs_df.shape[0])
total_pairs_with_drops = int(dropped_df.shape[0])
total_single_point_dropped = int(dropped_df['single_point_exceedances_dropped'].sum())

# Prepare top 5 runs summary (convert timestamps to ISO Asia/Shanghai)
top_n = min(5, total_runs_returned)
top_runs_lines = []
for i in range(top_n):
    r = runs_df.iloc[i]
    start_iso = pd.to_datetime(int(r['run_start_timestamp']), unit='s', utc=True).tz_convert(tz).isoformat()
    end_iso = pd.to_datetime(int(r['run_end_timestamp']), unit='s', utc=True).tz_convert(tz).isoformat()
    severity = r['severity_ratio']
    severity_str = f"{severity:.3f}" if pd.notna(severity) else "undefined (P95==0)"
    top_runs_lines.append(
        f"{i+1}. {r['cmdb_id']} | {r['kpi_name']}\n"
        f"     run_start: {start_iso}, run_end: {end_iso}, length: {int(r['run_length_in_points'])} points\n"
        f"     run_max_value: {float(r['run_max_value'])}, P95_value: {float(r['P95_value']) if pd.notna(r['P95_value']) else None}, severity_ratio: {severity_str}"
    )

# Identify pairs where P95 == 0 (severity undefined) in the returned runs
p95_zero_pairs = runs_df[runs_df['P95_value'] == 0.0][['cmdb_id','kpi_name','run_max_value']].drop_duplicates().to_dict('records')

# Example dropped single-point counts for top pairs (show up to 10)
dropped_examples = dropped_df.sort_values('single_point_exceedances_dropped', ascending=False).head(10).to_dict('records')

summary = (
    "Anomalous run summary (top-10 pairs analyzed, runs within 2022-03-21 03:00:00+08 to 03:29:00+08):\n\n"
    f"- Total multi-point anomalous runs returned: {total_runs_returned} (limited to 50 in output).\n"
    f"- Total examined pairs from top-10: {total_pairs_with_drops}. Single-point exceedances dropped (summed): {total_single_point_dropped}.\n\n"
    "Top anomalous runs (by severity_ratio):\n"
    + "\n".join(top_runs_lines) + "\n\n"
    "Notes:\n"
    "- The top run is for checkoutservice-0 / mesh.source.checkoutservice.jaeger-collector..., length 3, "
    "run_max_value 9.650525 vs P95 0.001575 -> severity_ratio ~6127x (very large).\n"
    "- Several runs (for checkoutservice-2 mesh.destination.frontend.checkoutservice...) have P95 == 0.0, "
    "so severity_ratio is undefined; these runs show large raw values (examples: run_max 785, 1350, 28, 1).\n"
    f"- Dropped single-point exceedances per pair (examples, up to 10 shown): {dropped_examples}.\n\n"
    "Interpretation guidance:\n"
    "- Multi-point runs (length >=2) are included; isolated single-point spikes were excluded from primary results and counted separately.\n"
    "- Extremely large severity_ratio (e.g., >1000) indicates run_max far above historical P95 for that (cmdb_id,kpi).\n"
    "- Cases with P95 == 0 should be inspected carefully (division undefined); consider whether the KPI is usually zero and whether absolute values matter.\n\n"
    "If you want the full run table or additional context (e.g., convert all run timestamps to ISO or inspect specific pairs), I can provide that next."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(             cmdb_id                                           kpi_name  run_start_timestamp  run_end_timestamp  run_length_in_points  run_max_value  P95_value  severity_ratio
0  checkoutservice-0  mesh.source.checkoutservice.jaeger-collector.i...           1647803520         1647803640                     3       9.650525   0.001575      6127.31746
1  checkoutservice-2  mesh.destination.frontend.checkoutservice.isti...           1647803460         1647803580                     3     785.000000   0.000000             NaN
2  checkoutservice-2  mesh.destination.frontend.checkoutservice.isti...           1647803460         1647803580                     3      28.000000   0.000000             NaN
3  checkoutservice-2  mesh.destination.frontend.checkoutservice.isti...           1647803460         1647803580                     3       1.000000   0.000000             NaN
4  checkoutservice-2  mesh.destination.frontend.checkoutservice.isti...           1647803460         1647803580                     3    1350.000000   0.000000             NaN,              cmdb_id                                           kpi_name  single_point_exceedances_dropped
0      cartservice-1         container.node-6.container_cpu_cfs_periods                                 3
1      cartservice-2       container.node-6.container_cpu_usage_seconds                                 3
2      cartservice-2        container.node-6.container_cpu_user_seconds                                 3
3  checkoutservice-0        container.node-6.container_cpu_user_seconds                                 3
4  checkoutservice-0  mesh.source.checkoutservice.jaeger-collector.i...                                 0
5  checkoutservice-2  container.node-5.container_network_transmit_MB...                                 3
6  checkoutservice-2  mesh.destination.frontend.checkoutservice.isti...                                 0
7  checkoutservice-2  mesh.destination.frontend.checkoutservice.isti...                                 0
8  checkoutservice-2  mesh.destination.frontend.checkoutservice.isti...                                 0
9  checkoutservice-2  mesh.destination.frontend.checkoutservice.isti...                                 0)```
```

In [ ]:
# In[5]:

import pandas as pd
import pytz
import numpy as np

tz = pytz.timezone('Asia/Shanghai')

# Reuse df_trace if present; otherwise load
try:
    df_trace
except NameError:
    df_trace = pd.read_csv('trace.csv')

# Ensure correct types
df_trace['timestamp'] = df_trace['timestamp'].astype('int64')
df_trace['value'] = pd.to_numeric(df_trace['value'], errors='coerce')

# 1) Identify pairs where trace_name contains the keywords
keywords_mask = df_trace['trace_name'].str.contains('error_rate|duration|row_count', case=False, na=False)
trace_pairs_df = df_trace[keywords_mask][['cmdb_id', 'trace_name']].drop_duplicates().reset_index(drop=True)

# 2) Compute global P95 for each (cmdb_id, trace_name) across entire trace.csv
p95_all = (
    df_trace
    .groupby(['cmdb_id', 'trace_name'], as_index=False)['value']
    .quantile(0.95)
    .rename(columns={'value': 'P95_value'})
)

# Restrict P95 to only the pairs of interest
p95_pairs = trace_pairs_df.merge(p95_all, on=['cmdb_id', 'trace_name'], how='left')

# 3) Filter trace rows to incident window
start_ts = 1647802800
end_ts = 1647804540
window_mask = (df_trace['timestamp'] >= start_ts) & (df_trace['timestamp'] <= end_ts)
df_window = df_trace[window_mask].copy()

# Also restrict to keyword pairs in window
df_window_pairs = df_window[df_window['trace_name'].str.contains('error_rate|duration|row_count', case=False, na=False)].copy()

# 4) For each pair compute aggregates within the window
# total points and max
window_agg = (
    df_window_pairs
    .groupby(['cmdb_id', 'trace_name'], as_index=False)
    .agg(total_points_in_window=('value', 'size'),
         max_value_in_window=('value', 'max'))
)

# Join P95 to window rows to detect exceedances
df_window_with_p95 = df_window_pairs.merge(p95_pairs, on=['cmdb_id', 'trace_name'], how='left')

# Flag exceedances (value > P95); if P95 is NaN, treat as no exceedance
df_window_with_p95['is_exceed'] = False
mask_p95_valid = df_window_with_p95['P95_value'].notna()
df_window_with_p95.loc[mask_p95_valid, 'is_exceed'] = df_window_with_p95.loc[mask_p95_valid, 'value'] > df_window_with_p95.loc[mask_p95_valid, 'P95_value']

# Aggregate exceedance counts and earliest anomalous timestamp per pair
exceed_agg = (
    df_window_with_p95[df_window_with_p95['is_exceed']]
    .groupby(['cmdb_id', 'trace_name'], as_index=False)
    .agg(count_points_exceeding_P95=('is_exceed', 'sum'),
         earliest_anomalous_timestamp=('timestamp', 'min'))
)

# Merge all per-pair results
pairs_agg = (
    p95_pairs
    .merge(window_agg, on=['cmdb_id', 'trace_name'], how='left')
    .merge(exceed_agg, on=['cmdb_id', 'trace_name'], how='left')
)

# Fill NaNs appropriately
pairs_agg['total_points_in_window'] = pairs_agg['total_points_in_window'].fillna(0).astype(int)
pairs_agg['count_points_exceeding_P95'] = pairs_agg['count_points_exceeding_P95'].fillna(0).astype(int)
pairs_agg['earliest_anomalous_timestamp'] = pairs_agg['earliest_anomalous_timestamp'].where(pairs_agg['earliest_anomalous_timestamp'].notna(), None)

# 5) Prepare top 20 pairs sorted by count_points_exceeding_P95 desc
top_pairs = pairs_agg.sort_values(by=['count_points_exceeding_P95', 'total_points_in_window'], ascending=[False, False]).head(20)

# Select and order required columns
top_pairs_df = top_pairs[['cmdb_id', 'trace_name', 'P95_value', 'total_points_in_window', 'count_points_exceeding_P95', 'earliest_anomalous_timestamp']].reset_index(drop=True)

# For service-level summary: aggregate count_points_exceeding_P95 per cmdb_id and earliest anomaly per service
service_agg = (
    top_pairs_df
    .groupby('cmdb_id', as_index=False)
    .agg(total_exceedances=('count_points_exceeding_P95', 'sum'),
         earliest_anomalous_timestamp=('earliest_anomalous_timestamp', lambda s: None if s.dropna().empty else int(s.dropna().min())))
    .sort_values(by='total_exceedances', ascending=False)
    .head(5)
)

# Convert earliest timestamps to ISO Asia/Shanghai if present
def to_iso(ts):
    if ts is None:
        return None
    return pd.to_datetime(int(ts), unit='s', utc=True).tz_convert(tz).isoformat()

service_agg['earliest_anomalous_iso_ASIA_SHANGHAI'] = service_agg['earliest_anomalous_timestamp'].apply(to_iso)

# Keep compact outputs: top_pairs_df (up to 20) and service_agg selected columns
service_summary_df = service_agg[['cmdb_id', 'total_exceedances', 'earliest_anomalous_iso_ASIA_SHANGHAI']]

top_pairs_df, service_summary_df

```
Out[5]:
```
```python
# Summarize trace-based localization results in plain English.
# Reuse previous variables top_pairs_df and service_summary_df if available; otherwise load minimal needed data.

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

try:
    top_pairs_df
    service_summary_df
except NameError:
    # Fallback: load and recompute compact aggregates (safe guard)
    df_trace = pd.read_csv('trace.csv')
    df_trace['timestamp'] = df_trace['timestamp'].astype('int64')
    df_trace['value'] = pd.to_numeric(df_trace['value'], errors='coerce')
    keywords_mask = df_trace['trace_name'].str.contains('error_rate|duration|row_count', case=False, na=False)
    trace_pairs_df = df_trace[keywords_mask][['cmdb_id', 'trace_name']].drop_duplicates().reset_index(drop=True)
    p95_all = (
        df_trace
        .groupby(['cmdb_id', 'trace_name'], as_index=False)['value']
        .quantile(0.95)
        .rename(columns={'value': 'P95_value'})
    )
    p95_pairs = trace_pairs_df.merge(p95_all, on=['cmdb_id', 'trace_name'], how='left')
    start_ts = 1647802800
    end_ts = 1647804540
    df_window = df_trace[(df_trace['timestamp'] >= start_ts) & (df_trace['timestamp'] <= end_ts)].copy()
    df_window_pairs = df_window[df_window['trace_name'].str.contains('error_rate|duration|row_count', case=False, na=False)].copy()
    window_agg = (
        df_window_pairs
        .groupby(['cmdb_id', 'trace_name'], as_index=False)
        .agg(total_points_in_window=('value', 'size'))
    )
    df_window_with_p95 = df_window_pairs.merge(p95_pairs, on=['cmdb_id', 'trace_name'], how='left')
    df_window_with_p95['is_exceed'] = False
    mask_p95_valid = df_window_with_p95['P95_value'].notna()
    df_window_with_p95.loc[mask_p95_valid, 'is_exceed'] = df_window_with_p95.loc[mask_p95_valid, 'value'] > df_window_with_p95.loc[mask_p95_valid, 'P95_value']
    exceed_agg = (
        df_window_with_p95[df_window_with_p95['is_exceed']]
        .groupby(['cmdb_id', 'trace_name'], as_index=False)
        .agg(count_points_exceeding_P95=('is_exceed', 'sum'),
             earliest_anomalous_timestamp=('timestamp', 'min'))
    )
    pairs_agg = (
        p95_pairs
        .merge(window_agg, on=['cmdb_id', 'trace_name'], how='left')
        .merge(exceed_agg, on=['cmdb_id', 'trace_name'], how='left')
    )
    pairs_agg['total_points_in_window'] = pairs_agg['total_points_in_window'].fillna(0).astype(int)
    pairs_agg['count_points_exceeding_P95'] = pairs_agg['count_points_exceeding_P95'].fillna(0).astype(int)
    pairs_agg['earliest_anomalous_timestamp'] = pairs_agg['earliest_anomalous_timestamp'].where(pairs_agg['earliest_anomalous_timestamp'].notna(), None)
    top_pairs_df = pairs_agg.sort_values(by=['count_points_exceeding_P95', 'total_points_in_window'], ascending=[False, False]).head(20)
    top_pairs_df = top_pairs_df[['cmdb_id', 'trace_name', 'P95_value', 'total_points_in_window', 'count_points_exceeding_P95', 'earliest_anomalous_timestamp']].reset_index(drop=True)
    service_agg = (
        top_pairs_df
        .groupby('cmdb_id', as_index=False)
        .agg(total_exceedances=('count_points_exceeding_P95', 'sum'),
             earliest_anomalous_timestamp=('earliest_anomalous_timestamp', lambda s: None if s.dropna().empty else int(s.dropna().min())))
        .sort_values(by='total_exceedances', ascending=False)
        .head(5)
    )
    def to_iso(ts):
        if ts is None:
            return None
        return pd.to_datetime(int(ts), unit='s', utc=True).tz_convert(tz).isoformat()
    service_agg['earliest_anomalous_iso_ASIA_SHANGHAI'] = service_agg['earliest_anomalous_timestamp'].apply(to_iso)
    service_summary_df = service_agg[['cmdb_id', 'total_exceedances', 'earliest_anomalous_iso_ASIA_SHANGHAI']]

# Build plain-English summary
# Top services by aggregated trace exceedances
services = service_summary_df.to_dict('records')
# Top trace pairs (show up to 5 examples)
top_pairs_examples = top_pairs_df.head(5).to_dict('records')

summary_lines = []
summary_lines.append("Trace analysis (window 2022-03-21 03:00:00+08 to 03:29:00+08):")
summary_lines.append("")
summary_lines.append("1) Metrics considered: trace metrics whose names include 'error_rate', 'duration', or 'row_count'.")
summary_lines.append("2) Global P95 thresholds were computed per (cmdb_id, trace_name) using all historical rows before window filtering.")
summary_lines.append("")
summary_lines.append("3) Top services by aggregated number of trace values exceeding their P95 within the incident window (top 5):")
for s in services:
    summary_lines.append(f"   - {s['cmdb_id']}: total_exceedances={int(s['total_exceedances'])}, earliest_anomaly={s['earliest_anomalous_iso_ASIA_SHANGHAI']}")
summary_lines.append("")
summary_lines.append("Interpretation / localization:")
summary_lines.append("   - The downstream-most faulty service appears to be adservice-1 (highest aggregated exceedances: 14),")
summary_lines.append("     followed by adservice-0 and adservice-2 (each with 12 exceedances).")
summary_lines.append("   - Earliest anomalous timestamps per service: adservice-0 at 2022-03-21T03:00:00+08,")
summary_lines.append("     adservice-1 at 2022-03-21T03:08:00+08, adservice-2 at 2022-03-21T03:09:00+08.")
summary_lines.append("")
summary_lines.append("4) Example affected trace metrics (top 5 pairs by exceedance count):")
for p in top_pairs_examples:
    et = p['earliest_anomalous_timestamp']
    et_iso = None if et is None else pd.to_datetime(int(et), unit='s', utc=True).tz_convert(tz).isoformat()
    summary_lines.append(f"   - {p['cmdb_id']} | {p['trace_name']}: P95={p['P95_value']}, total_points={p['total_points_in_window']}, exceedances={int(p['count_points_exceeding_P95'])}, earliest={et_iso}")
summary_lines.append("")
summary_lines.append("Summary conclusion: trace signals point to adservice (especially adservice-1) as the downstream-most component showing the most frequent deviations above historical P95 during the incident window. Recommend focusing investigation on the adservice pods and their traces (latency/error-rate/row-count) first.")

summary = "\n".join(summary_lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(         cmdb_id                            trace_name  P95_value  total_points_in_window  count_points_exceeding_P95  earliest_anomalous_timestamp
0    adservice-0   trace.from_frontend-0.duration_mean   0.000029                      30                           2                  1.647803e+09
1    adservice-0    trace.from_frontend-0.duration_p95   0.000070                      30                           2                  1.647803e+09
2    adservice-0   trace.from_frontend-1.duration_mean   0.000023                      30                           2                  1.647803e+09
3    adservice-0    trace.from_frontend-1.duration_p95   0.000047                      30                           2                  1.647803e+09
4    adservice-0   trace.from_frontend-2.duration_mean   0.000023                      30                           2                  1.647803e+09
5    adservice-0    trace.from_frontend-2.duration_p95   0.000032                      30                           2                  1.647803e+09
6    adservice-1   trace.from_frontend-0.duration_mean   0.000021                      30                           2                  1.647803e+09
7    adservice-1    trace.from_frontend-0.duration_p95   0.000042                      30                           2                  1.647803e+09
8    adservice-1       trace.from_frontend-0.row_count   9.550000                      30                           2                  1.647803e+09
9    adservice-1   trace.from_frontend-1.duration_mean   0.000024                      30                           2                  1.647803e+09
10   adservice-1    trace.from_frontend-1.duration_p95   0.000061                      30                           2                  1.647803e+09
11   adservice-1   trace.from_frontend-2.duration_mean   0.000031                      30                           2                  1.647803e+09
12   adservice-1    trace.from_frontend-2.duration_p95   0.000057                      30                           2                  1.647803e+09
13   adservice-2   trace.from_frontend-0.duration_mean   0.000020                      30                           2                  1.647803e+09
14   adservice-2    trace.from_frontend-0.duration_p95   0.000042                      30                           2                  1.647803e+09
15   adservice-2    trace.from_frontend-1.duration_p95   0.000029                      30                           2                  1.647804e+09
16   adservice-2   trace.from_frontend-2.duration_mean   0.000021                      30                           2                  1.647804e+09
17   adservice-2    trace.from_frontend-2.duration_p95   0.000033                      30                           2                  1.647804e+09
18   adservice-2       trace.from_frontend-2.row_count   3.550000                      30                           2                  1.647803e+09
19  adservice2-0  trace.from_frontend2-0.duration_mean   0.000088                      30                           2                  1.647803e+09,         cmdb_id  total_exceedances earliest_anomalous_iso_ASIA_SHANGHAI
1   adservice-1                 14            2022-03-21T03:08:00+08:00
0   adservice-0                 12            2022-03-21T03:00:00+08:00
2   adservice-2                 12            2022-03-21T03:09:00+08:00
3  adservice2-0                  2            2022-03-21T03:00:00+08:00)```
```

In [ ]:
# In[6]:

import pandas as pd
import pytz
import numpy as np

tz = pytz.timezone('Asia/Shanghai')

# Reuse existing loaded DataFrames if present; otherwise load from CSV
try:
    df_error
except NameError:
    df_error = pd.read_csv('error_logs.csv')
try:
    df_log
except NameError:
    df_log = pd.read_csv('log.csv')

# Ensure types
df_error['timestamp'] = df_error['timestamp'].astype('int64')
df_error['message'] = df_error['message'].astype(str)

df_log['timestamp'] = df_log['timestamp'].astype('int64')
df_log['value'] = pd.to_numeric(df_log['value'], errors='coerce')

# Incident window
start_ts = 1647802800
end_ts = 1647804540

# 1) error_logs.csv filtering
err_time_mask = (df_error['timestamp'] >= start_ts) & (df_error['timestamp'] <= end_ts)
err_msg_mask = df_error['message'].str.contains(r'POST /cart/checkout|500|via_upstream', case=False, na=False)
err_filtered = df_error[err_time_mask & err_msg_mask].copy()

# total count
error_total_count = int(len(err_filtered))

# per-cmdb_id counts and earliest timestamp per cmdb_id
if not err_filtered.empty:
    per_cmdb = (
        err_filtered
        .groupby('cmdb_id', as_index=False)
        .agg(count=('message', 'size'),
             earliest_timestamp=('timestamp', 'min'))
        .sort_values('count', ascending=False)
    )
    # ensure earliest_timestamp is int
    per_cmdb['earliest_timestamp'] = per_cmdb['earliest_timestamp'].astype('Int64')
    per_cmdb = per_cmdb[['cmdb_id', 'count', 'earliest_timestamp']]
else:
    per_cmdb = pd.DataFrame(columns=['cmdb_id', 'count', 'earliest_timestamp'])

# up to 10 distinct message samples with counts (full message text)
msg_samples = (
    err_filtered['message']
    .value_counts()
    .head(10)
    .rename_axis('message')
    .reset_index(name='count')
)

# 2) log.csv filtering for log.error_count in window and specified cmdbs
targets = [
    "frontend-0","frontend-1","frontend-2",
    "checkoutservice-0","checkoutservice-1","checkoutservice-2",
    "cartservice-0","cartservice-1","cartservice-2",
    "adservice-0","adservice-1","adservice-2"
]

log_time_mask = (df_log['timestamp'] >= start_ts) & (df_log['timestamp'] <= end_ts)
log_name_mask = df_log['log_name'] == 'log.error_count'
log_filtered = df_log[log_time_mask & log_name_mask & df_log['cmdb_id'].isin(targets)].copy()

# Prepare per-cmdb aggregates; ensure all targets present
if not log_filtered.empty:
    # total_rows, sum_of_value, count_minutes_with_value_gt_0, earliest_timestamp_where_value_gt_0, max_value
    agg = (
        log_filtered
        .groupby('cmdb_id', as_index=False)
        .agg(total_rows=('value', 'size'),
             sum_of_value=('value', 'sum'),
             count_minutes_with_value_gt_0=('value', lambda s: int((s > 0).sum())),
             earliest_ts_value_gt_0=('timestamp', lambda s: int(s[s[ (log_filtered.loc[s.index,'value'] > 0) ] .index].min()) if (log_filtered.loc[s.index,'value'] > 0).any() else pd.NA),
             max_value=('value', 'max'))
    )
    # The lambda above is a bit complex; recompute earliest properly per group for correctness:
    # Recompute earliest where value>0 per group
    earliest_list = []
    for name, group in log_filtered.groupby('cmdb_id'):
        gt = group[group['value'] > 0]
        earliest_list.append((name, int(gt['timestamp'].min()) if not gt.empty else pd.NA))
    earliest_df = pd.DataFrame(earliest_list, columns=['cmdb_id', 'earliest_timestamp_where_value_gt_0'])
    agg = agg.drop(columns=['earliest_ts_value_gt_0']).merge(earliest_df, on='cmdb_id', how='left')
else:
    agg = pd.DataFrame(columns=['cmdb_id','total_rows','sum_of_value','count_minutes_with_value_gt_0','earliest_timestamp_where_value_gt_0','max_value'])

# Ensure all targets included (fill missing with zeros/NA)
targets_df = pd.DataFrame({'cmdb_id': targets})
agg_full = targets_df.merge(agg, on='cmdb_id', how='left')
agg_full['total_rows'] = agg_full['total_rows'].fillna(0).astype(int)
agg_full['sum_of_value'] = agg_full['sum_of_value'].fillna(0).astype(float)
agg_full['count_minutes_with_value_gt_0'] = agg_full['count_minutes_with_value_gt_0'].fillna(0).astype(int)
# earliest timestamp keep as Int64 nullable
agg_full['earliest_timestamp_where_value_gt_0'] = agg_full['earliest_timestamp_where_value_gt_0'].where(agg_full['earliest_timestamp_where_value_gt_0'].notna(), None)
agg_full['max_value'] = agg_full['max_value'].where(agg_full['max_value'].notna(), 0.0).astype(float)

# Sort by sum_of_value desc and limit top 20
logs_summary = agg_full.sort_values(by='sum_of_value', ascending=False).head(20).reset_index(drop=True)

# Compact JSON-serializable outputs: error_total_count (int), per_cmdb (DataFrame), msg_samples (DataFrame), logs_summary (DataFrame)
error_total_count, per_cmdb.head(20), msg_samples, logs_summary[['cmdb_id','total_rows','sum_of_value','count_minutes_with_value_gt_0','earliest_timestamp_where_value_gt_0','max_value']].head(20)

```
Out[6]:
```
Summary (compact, plain English):

1) error_logs.csv (filtered to 2022-03-21 03:00:00+08 — 03:29:00+08 and messages containing "POST /cart/checkout" or "500" or "via_upstream")
- Total matching rows: 7
- Per-cmdb_id counts and earliest timestamps (integer seconds):
  - frontend-0: count = 3, earliest_timestamp = 1647803487
  - frontend-1: count = 3, earliest_timestamp = 1647803508
  - frontend-2: count = 1, earliest_timestamp = 1647803492
- Distinct message samples (up to 10): 7 distinct samples, each appears once. They are HTTP access/error lines for:
  - "POST /cart/checkout HTTP/1.1" 500 - via_upstream ... (several similar full access log entries with different request IDs / upstream addresses)
  (All 7 matched messages are variants of the POST /cart/checkout 500 via_upstream access log lines; each sample count = 1.)

2) log.csv (log_name == 'log.error_count' within the same time window) — per requested component set
(Fields: total_rows, sum_of_value, count_minutes_with_value_gt_0, earliest_timestamp_where_value_gt_0, max_value)
Sorted by sum_of_value (top entries shown):

- frontend-0: total_rows=30, sum_of_value=6.0, count_minutes_with_value_gt_0=2, earliest_timestamp_where_value_gt_0=1647803460, max_value=4.0
- frontend-1: total_rows=30, sum_of_value=6.0, count_minutes_with_value_gt_0=2, earliest_timestamp_where_value_gt_0=1647803460, max_value=4.0
- frontend-2: total_rows=30, sum_of_value=2.0, count_minutes_with_value_gt_0=1, earliest_timestamp_where_value_gt_0=1647803460, max_value=2.0
- checkoutservice-0: total_rows=30, sum_of_value=0.0, count_minutes_with_value_gt_0=0, earliest_timestamp_where_value_gt_0=None, max_value=0.0
- checkoutservice-1: total_rows=22, sum_of_value=0.0, count_minutes_with_value_gt_0=0, earliest_timestamp_where_value_gt_0=None, max_value=0.0
- checkoutservice-2: total_rows=27, sum_of_value=0.0, count_minutes_with_value_gt_0=0, earliest_timestamp_where_value_gt_0=None, max_value=0.0
- cartservice-0: total_rows=30, sum_of_value=0.0, count_minutes_with_value_gt_0=0, earliest_timestamp_where_value_gt_0=None, max_value=0.0
- cartservice-1: total_rows=30, sum_of_value=0.0, count_minutes_with_value_gt_0=0, earliest_timestamp_where_value_gt_0=None, max_value=0.0
- cartservice-2: total_rows=30, sum_of_value=0.0, count_minutes_with_value_gt_0=0, earliest_timestamp_where_value_gt_0=None, max_value=0.0
- adservice-0: total_rows=30, sum_of_value=0.0, count_minutes_with_value_gt_0=0, earliest_timestamp_where_value_gt_0=None, max_value=0.0
- adservice-1: total_rows=30, sum_of_value=0.0, count_minutes_with_value_gt_0=0, earliest_timestamp_where_value_gt_0=None, max_value=0.0
- adservice-2: total_rows=30, sum_of_value=0.0, count_minutes_with_value_gt_0=0, earliest_timestamp_where_value_gt_0=None, max_value=0.0

Interpretation / key takeaways (concise):
- Error logs and normalized log.error_count both point to the frontend pods as the primary locations showing errors during the incident window.
- The frontends (frontend-0 and frontend-1) show the highest error_count sums (6 each) and multiple error_log entries reporting "POST /cart/checkout" HTTP 500 responses (via_upstream).
- Backend services listed (checkoutservice, cartservice, adservice) show zero aggregated error_count in log.error_count during the window (per the available normalized log.csv).
- Next investigation steps: inspect frontend-0 and frontend-1 (their proxy/upstream connectivity, upstream responses, and recent config/deployments), and correlate the detailed full access/error log lines (the POST /cart/checkout 500 via_upstream samples) with upstream endpoints they reference.

The original code execution output of IPython Kernel is also provided below for reference:

(7,       cmdb_id  count  earliest_timestamp
0  frontend-0      3          1647803487
1  frontend-1      3          1647803508
2  frontend-2      1          1647803492,                                              message  count
0  "POST /cart/checkout HTTP/1.1" 500 - via_upstr...      1
1  "POST /cart/checkout HTTP/1.1" 500 - via_upstr...      1
2  "POST /cart/checkout HTTP/1.1" 500 - via_upstr...      1
3  "POST /cart/checkout HTTP/1.1" 500 - via_upstr...      1
4  "POST /cart/checkout HTTP/1.1" 500 - via_upstr...      1
5  "POST /cart/checkout HTTP/1.1" 500 - via_upstr...      1
6  "POST /cart/checkout HTTP/1.1" 500 - via_upstr...      1,               cmdb_id  total_rows  sum_of_value  count_minutes_with_value_gt_0 earliest_timestamp_where_value_gt_0  max_value
0          frontend-0          30           6.0                              2                          1647803460        4.0
1          frontend-1          30           6.0                              2                          1647803460        4.0
2          frontend-2          30           2.0                              1                          1647803460        2.0
3   checkoutservice-0          30           0.0                              0                                None        0.0
4   checkoutservice-1          22           0.0                              0                                None        0.0
5   checkoutservice-2          27           0.0                              0                                None        0.0
6       cartservice-0          30           0.0                              0                                None        0.0
7       cartservice-1          30           0.0                              0                                None        0.0
8       cartservice-2          30           0.0                              0                                None        0.0
9         adservice-0          30           0.0                              0                                None        0.0
10        adservice-1          30           0.0                              0                                None        0.0
11        adservice-2          30           0.0                              0                                None        0.0)```
```

In [ ]:
# In[7]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Load or reuse metric dataframe
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv')

# Ensure correct types
df_metric['timestamp'] = df_metric['timestamp'].astype('int64')
df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')
df_metric['kpi_name'] = df_metric['kpi_name'].astype(str)
df_metric['cmdb_id'] = df_metric['cmdb_id'].astype(str)

# Targets and keyword pattern (case-insensitive)
targets = [
    "checkoutservice-0","checkoutservice-1","checkoutservice-2",
    "cartservice-0","cartservice-1","cartservice-2"
]
kw_pattern = r'mesh\.|mesh\.destination|mesh\.source|istio|network'

# Identify relevant pairs within the filtered set (for later limiting)
mask_targets = df_metric['cmdb_id'].isin(targets)
mask_kw = df_metric['kpi_name'].str.contains(kw_pattern, case=False, na=False)
filtered_pairs_df = df_metric.loc[mask_targets & mask_kw, ['cmdb_id', 'kpi_name']].drop_duplicates().reset_index(drop=True)

# Compute global P95 per (cmdb_id, kpi_name) using entire metric.csv (rule 9)
p95_all = (
    df_metric
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)['value']
    .quantile(0.95)
    .rename(columns={'value': 'P95_value'})
)

# Restrict to relevant pairs (if none, result will be empty)
p95_pairs = filtered_pairs_df.merge(p95_all, on=['cmdb_id', 'kpi_name'], how='left')

# Incident window filter
start_ts = 1647802800
end_ts = 1647804540
window_mask = (df_metric['timestamp'] >= start_ts) & (df_metric['timestamp'] <= end_ts)
df_window = df_metric[window_mask].copy()

# Restrict window rows to targets and keyword KPIs
df_window_pairs = df_window[df_window['cmdb_id'].isin(targets) & df_window['kpi_name'].str.contains(kw_pattern, case=False, na=False)].copy()

# Compute total points and max value in window per pair
if not df_window_pairs.empty:
    window_agg = (
        df_window_pairs
        .groupby(['cmdb_id', 'kpi_name'], as_index=False)
        .agg(total_points_in_window=('value', 'size'),
             max_value_in_window=('value', 'max'))
    )
else:
    window_agg = pd.DataFrame(columns=['cmdb_id','kpi_name','total_points_in_window','max_value_in_window'])

# Join window rows with P95 to detect exceedances
df_win_with_p95 = df_window_pairs.merge(p95_pairs, on=['cmdb_id', 'kpi_name'], how='left')

# Flag exceedances where P95 is valid (non-null)
df_win_with_p95['is_exceed'] = False
mask_p95_valid = df_win_with_p95['P95_value'].notna()
df_win_with_p95.loc[mask_p95_valid, 'is_exceed'] = df_win_with_p95.loc[mask_p95_valid, 'value'] > df_win_with_p95.loc[mask_p95_valid, 'P95_value']

# Aggregate exceedance counts and earliest anomalous timestamp per pair
if df_win_with_p95['is_exceed'].any():
    exceed_agg = (
        df_win_with_p95[df_win_with_p95['is_exceed']]
        .groupby(['cmdb_id', 'kpi_name'], as_index=False)
        .agg(count_points_exceeding_P95=('is_exceed', 'sum'),
             earliest_anomalous_timestamp=('timestamp', 'min'))
    )
else:
    exceed_agg = pd.DataFrame(columns=['cmdb_id','kpi_name','count_points_exceeding_P95','earliest_anomalous_timestamp'])

# Merge results
agg = (
    p95_pairs
    .merge(window_agg, on=['cmdb_id', 'kpi_name'], how='left')
    .merge(exceed_agg, on=['cmdb_id', 'kpi_name'], how='left')
)

# Fill NaNs appropriately
agg['total_points_in_window'] = agg['total_points_in_window'].fillna(0).astype(int)
agg['count_points_exceeding_P95'] = agg['count_points_exceeding_P95'].fillna(0).astype(int)
agg['earliest_anomalous_timestamp'] = agg['earliest_anomalous_timestamp'].where(agg['earliest_anomalous_timestamp'].notna(), None)
agg['max_value_in_window'] = agg['max_value_in_window'].where(agg['max_value_in_window'].notna(), np.nan)
agg['P95_value'] = agg['P95_value'].where(agg['P95_value'].notna(), np.nan)

# Filter to pairs with at least one exceedance and sort as requested
result_df = agg[agg['count_points_exceeding_P95'] > 0].copy()
result_df = result_df.sort_values(by=['count_points_exceeding_P95', 'max_value_in_window'], ascending=[False, False]).head(20)

# Select and order columns for compact JSON-serializable output
result_df = result_df[['cmdb_id', 'kpi_name', 'P95_value', 'total_points_in_window', 'count_points_exceeding_P95', 'earliest_anomalous_timestamp', 'max_value_in_window']].reset_index(drop=True)

# Final variable to display (compact)
result_df

```
Out[7]:
```
Summary (compact, plain English):

- Scope: analyzed mesh/istio/network-related KPIs for checkoutservice-* and cartservice-* during the incident window (2022-03-21 ~03:00–03:29+08). Only (cmdb_id, kpi_name) pairs with at least one value > their global P95 are shown (top 20).

Key findings (top anomalies):
1. checkoutservice-2 shows the heaviest anomalies (each with 3 exceedances):
   - mesh.destination.frontend.checkoutservice.* (multiple KPI variants) — P95 reported as 0.0, max values 1350, 785, 28 (three separate KPI entries). Earliest anomalous points occur around the start of the window (~03:00–03:09+08).
   - container.node-5.container_network_transmit_MB... — P95 ~2.327, max ~2.347 (3 exceedances).
   - mesh.source.checkoutservice.productcatalogservice... — P95 ~1170, max ~1425 (2 exceedances in another pair).
   - Several checkoutservice-2 mesh/istio KPIs have P95 == 0, indicating historically-zero baseline and sudden non-zero spikes.

2. checkoutservice-0 also shows notable anomalies:
   - mesh.source.checkoutservice.jaeger-collector... — P95 ~0.001575, max ~9.6505 (3 exceedances).
   - mesh.destination.frontend.checkoutservice.* — P95 == 0, max values up to 1450 (2–3 exceedances across entries).
   - mesh.source.checkoutservice.productcatalogservice... and network transmit KPIs show exceedances with large max values.

3. checkoutservice-1 shows mesh.destination.frontend.* anomalies (2 exceedances) including a very large max (2125) for one KPI (P95==0).

4. cartservice side:
   - cartservice-1: mesh.source.cartservice.redis-cart.istio_tcp_* (P95 1391, max ~1522.5) — 2 exceedances.
   - cartservice-0 and cartservice-1 show elevated container.node-6 network receive/transmit packet metrics (2 exceedances each, max values in the hundreds).

Interpretation / recommended focus:
- The most impacted service appears to be checkoutservice (especially checkoutservice-2 and checkoutservice-0), with multiple mesh/istio destination/source KPIs spiking well above historical P95 — including many KPIs whose historical P95 is zero (i.e., normally zero, now non-zero spikes).
- These mesh.destination/front-end and mesh.source metrics indicate abnormal traffic/latency/throughput or proxy/istio behavior between frontend and checkoutservice during the incident window. The very large raw maxima (e.g., 1350, 1450, 2125) suggest heavy request/byte counts or abnormal metrics on those mesh KPIs.
- Cartservice shows some network metric anomalies (redis-cart istio TCP KPI and node network packet metrics) but fewer exceedances than checkoutservice-2.

Actionable next steps:
- Investigate checkoutservice-2 and checkoutservice-0 mesh/istio endpoints: inspect Istio proxy logs, destination rules, and upstream call patterns from frontend -> checkoutservice during 03:00–03:10+08.
- Correlate the mesh spikes (P95==0 cases) with the frontend HTTP 500s seen earlier to confirm whether checkoutservice mesh anomalies caused the frontend errors.
- Check node-level network/transmit metrics on nodes hosting checkoutservice and cartservice pods for packet drops/throttling.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                           kpi_name    P95_value  total_points_in_window  count_points_exceeding_P95  earliest_anomalous_timestamp  max_value_in_window
0   checkoutservice-2  mesh.destination.frontend.checkoutservice.isti...     0.000000                      30                           3                  1.647803e+09          1350.000000
1   checkoutservice-2  mesh.destination.frontend.checkoutservice.isti...     0.000000                      30                           3                  1.647803e+09           785.000000
2   checkoutservice-2  mesh.destination.frontend.checkoutservice.isti...     0.000000                      30                           3                  1.647803e+09            28.000000
3   checkoutservice-0  mesh.source.checkoutservice.jaeger-collector.i...     0.001575                      30                           3                  1.647804e+09             9.650525
4   checkoutservice-2  container.node-5.container_network_transmit_MB...     2.327476                      30                           3                  1.647803e+09             2.347293
5   checkoutservice-2  mesh.source.checkoutservice.jaeger-collector.i...     0.008300                      30                           3                  1.647803e+09             1.557500
6   checkoutservice-2  mesh.destination.frontend.checkoutservice.isti...     0.000000                      30                           3                  1.647803e+09             1.000000
7   checkoutservice-1  mesh.destination.frontend.checkoutservice.isti...     0.000000                      30                           2                  1.647803e+09          2125.000000
8       cartservice-1  mesh.source.cartservice.redis-cart.istio_tcp_s...  1391.000000                      30                           2                  1.647803e+09          1522.500000
9   checkoutservice-0  mesh.destination.frontend.checkoutservice.isti...     0.000000                      30                           2                  1.647804e+09          1450.000000
10  checkoutservice-2  mesh.source.checkoutservice.productcatalogserv...  1170.000000                      30                           2                  1.647803e+09          1425.000000
11  checkoutservice-0  mesh.source.checkoutservice.productcatalogserv...  1125.000000                      30                           2                  1.647803e+09          1185.000000
12  checkoutservice-1  mesh.destination.frontend.checkoutservice.isti...     0.000000                      30                           2                  1.647803e+09          1177.500000
13  checkoutservice-0  mesh.destination.frontend.checkoutservice.isti...     0.000000                      30                           2                  1.647804e+09           785.000000
14      cartservice-1  container.node-6.container_network_receive_pac...   523.500000                      30                           2                  1.647804e+09           583.000000
15      cartservice-0  container.node-6.container_network_receive_pac...   510.500000                      30                           2                  1.647804e+09           536.000000
16      cartservice-1  container.node-6.container_network_transmit_pa...   379.500000                      30                           2                  1.647804e+09           459.000000
17  checkoutservice-1  container.node-6.container_network_transmit_pa...   387.500000                      30                           2                  1.647804e+09           450.500000
18  checkoutservice-0  container.node-6.container_network_transmit_pa...   400.000000                      30                           2                  1.647804e+09           418.000000
19  checkoutservice-2  container.node-5.container_network_transmit_pa...   346.500000                      30                           2                  1.647803e+09           415.833333```
```

In [ ]:
# In[8]:

import pandas as pd
import numpy as np
import pytz

# Reuse existing df_trace if available, else load
try:
    df_trace
except NameError:
    df_trace = pd.read_csv('trace.csv')

# Ensure types and timezone variable
df_trace['timestamp'] = df_trace['timestamp'].astype('int64')
df_trace['value'] = pd.to_numeric(df_trace['value'], errors='coerce')
tz = pytz.timezone('Asia/Shanghai')

# Incident window
start_ts = 1647802800
end_ts = 1647804540

# 1) Select pairs where trace_name contains 'from_frontend' AND 'error_rate' (case-insensitive)
mask_from_frontend = df_trace['trace_name'].str.contains('from_frontend', case=False, na=False)
mask_error_rate = df_trace['trace_name'].str.contains('error_rate', case=False, na=False)
mask_pairs = mask_from_frontend & mask_error_rate

trace_pairs_df = df_trace.loc[mask_pairs, ['cmdb_id', 'trace_name']].drop_duplicates().reset_index(drop=True)

# 2) Compute global P95 per (cmdb_id, trace_name) using entire df_trace
p95_all = (
    df_trace
    .groupby(['cmdb_id', 'trace_name'], as_index=False)['value']
    .quantile(0.95)
    .rename(columns={'value': 'P95_value'})
)

# Restrict P95 to only the pairs of interest
p95_pairs = trace_pairs_df.merge(p95_all, on=['cmdb_id', 'trace_name'], how='left')

# 3) Filter trace rows to incident window
df_window = df_trace[(df_trace['timestamp'] >= start_ts) & (df_trace['timestamp'] <= end_ts)].copy()

# Restrict window rows to the selected pairs (from_frontend & error_rate)
df_window_pairs = df_window[mask_pairs.loc[df_window.index].values].copy()  # align mask to df_window

# 4) For each pair compute aggregates within the window
# total points and max value
if not df_window_pairs.empty:
    window_agg = (
        df_window_pairs
        .groupby(['cmdb_id', 'trace_name'], as_index=False)
        .agg(total_points_in_window=('value', 'size'),
             max_value_in_window=('value', 'max'))
    )
else:
    window_agg = pd.DataFrame(columns=['cmdb_id','trace_name','total_points_in_window','max_value_in_window'])

# Join window rows with p95_pairs to evaluate exceedances
df_win_with_p95 = df_window_pairs.merge(p95_pairs, on=['cmdb_id', 'trace_name'], how='left')

# Flag exceedances only when P95 is present
df_win_with_p95['is_exceed'] = False
mask_p95_valid = df_win_with_p95['P95_value'].notna()
df_win_with_p95.loc[mask_p95_valid, 'is_exceed'] = df_win_with_p95.loc[mask_p95_valid, 'value'] > df_win_with_p95.loc[mask_p95_valid, 'P95_value']

# Aggregate exceedance counts and earliest anomalous timestamp per pair
if df_win_with_p95['is_exceed'].any():
    exceed_agg = (
        df_win_with_p95[df_win_with_p95['is_exceed']]
        .groupby(['cmdb_id', 'trace_name'], as_index=False)
        .agg(count_points_exceeding_P95=('is_exceed', 'sum'),
             earliest_anomalous_timestamp=('timestamp', 'min'))
    )
else:
    exceed_agg = pd.DataFrame(columns=['cmdb_id','trace_name','count_points_exceeding_P95','earliest_anomalous_timestamp'])

# Merge all per-pair results
pairs_agg = (
    p95_pairs
    .merge(window_agg, on=['cmdb_id', 'trace_name'], how='left')
    .merge(exceed_agg, on=['cmdb_id', 'trace_name'], how='left')
)

# Fill and normalize columns
pairs_agg['total_points_in_window'] = pairs_agg['total_points_in_window'].fillna(0).astype(int)
pairs_agg['count_points_exceeding_P95'] = pairs_agg['count_points_exceeding_P95'].fillna(0).astype(int)
pairs_agg['earliest_anomalous_timestamp'] = pairs_agg['earliest_anomalous_timestamp'].where(pairs_agg['earliest_anomalous_timestamp'].notna(), None)
pairs_agg['max_value_in_window'] = pairs_agg['max_value_in_window'].where(pairs_agg['max_value_in_window'].notna(), np.nan)
pairs_agg['P95_value'] = pairs_agg['P95_value'].where(pairs_agg['P95_value'].notna(), np.nan)

# 5) Aggregate per cmdb_id: sum of exceedances and earliest anomalous timestamp across its pairs
service_agg = (
    pairs_agg.groupby('cmdb_id', as_index=False)
    .agg(total_exceedances=('count_points_exceeding_P95', 'sum'),
         earliest_anomalous_timestamp_per_service=('earliest_anomalous_timestamp', lambda s: None if s.dropna().empty else int(s.dropna().min())))
    .sort_values(by='total_exceedances', ascending=False)
)

# 6) Prepare final outputs
top_pairs = pairs_agg.sort_values(by=['count_points_exceeding_P95', 'max_value_in_window'], ascending=[False, False]).head(20).reset_index(drop=True)
# select requested columns
top_pairs_df = top_pairs[['cmdb_id', 'trace_name', 'P95_value', 'total_points_in_window', 'count_points_exceeding_P95', 'earliest_anomalous_timestamp', 'max_value_in_window']]

service_summary_df = service_agg[['cmdb_id', 'total_exceedances', 'earliest_anomalous_timestamp_per_service']].head(10)

# Display compact results (DataFrames only)
top_pairs_df, service_summary_df

```
Out[8]:
```
Summary — targeted trace analysis for frontend-originated requests (trace_name contains "from_frontend" and "error_rate"):

- What was tested:
  - All (cmdb_id, trace_name) pairs where trace_name contains "from_frontend" and "error_rate".
  - Global P95 computed per pair using all history.
  - Window examined: 2022-03-21 03:00:00+08 through 03:29:00+08 (timestamps 1647802800–1647804540).
  - For each pair we checked total points in window, count of points > P95, earliest anomalous timestamp, and max value in window.

- Key result (compact):
  - Every examined (cmdb_id, trace_name) pair had:
    - P95_value = 0.0
    - total_points_in_window = 30
    - count_points_exceeding_P95 = 0
    - earliest_anomalous_timestamp = None
    - max_value_in_window = 0.0
  - Aggregated per service: all services (adservice, cartservice, checkoutservice variants shown) have total_exceedances = 0 and no earliest anomaly timestamp.

- Interpretation:
  - No frontend-originated trace error_rate metric exceeded its global P95 during the incident window. In other words, trace error_rate signals for "from_frontend" did not indicate anomalous error-rate rises for downstream services in this window.
  - Note: P95 is zero (historical baseline zero). Here max values are also zero, so there were no non-zero error_rate samples to trigger exceedance.

- Recommendation / next steps:
  - Since frontend-originated trace error_rate did not surface a downstream faulty service, investigate other trace signals and telemetry already flagged:
    - Trace duration and row_count metrics (some duration metrics earlier showed deviations).
    - Metric-level mesh/istio and network anomalies (checkoutservice and cartservice mesh KPIs showed spikes).
    - Frontend error logs and normalized log.error_count (frontend-0 and frontend-1 reported HTTP 500s).
  - Correlate the frontend 500s and mesh/checkoutservice spikes (checkoutservice-2/0) to identify the downstream root cause; focus on mesh/istio and checkoutservice pods next.

The original code execution output of IPython Kernel is also provided below for reference:

(           cmdb_id                         trace_name  P95_value  total_points_in_window  count_points_exceeding_P95 earliest_anomalous_timestamp  max_value_in_window
0      adservice-0   trace.from_frontend-0.error_rate        0.0                      30                           0                         None                  0.0
1      adservice-0   trace.from_frontend-1.error_rate        0.0                      30                           0                         None                  0.0
2      adservice-0   trace.from_frontend-2.error_rate        0.0                      30                           0                         None                  0.0
3      adservice-1   trace.from_frontend-0.error_rate        0.0                      30                           0                         None                  0.0
4      adservice-1   trace.from_frontend-1.error_rate        0.0                      30                           0                         None                  0.0
5      adservice-1   trace.from_frontend-2.error_rate        0.0                      30                           0                         None                  0.0
6      adservice-2   trace.from_frontend-0.error_rate        0.0                      30                           0                         None                  0.0
7      adservice-2   trace.from_frontend-1.error_rate        0.0                      30                           0                         None                  0.0
8      adservice-2   trace.from_frontend-2.error_rate        0.0                      30                           0                         None                  0.0
9     adservice2-0  trace.from_frontend2-0.error_rate        0.0                      30                           0                         None                  0.0
10   cartservice-0   trace.from_frontend-0.error_rate        0.0                      30                           0                         None                  0.0
11   cartservice-0   trace.from_frontend-1.error_rate        0.0                      30                           0                         None                  0.0
12   cartservice-0   trace.from_frontend-2.error_rate        0.0                      30                           0                         None                  0.0
13   cartservice-1   trace.from_frontend-0.error_rate        0.0                      30                           0                         None                  0.0
14   cartservice-1   trace.from_frontend-1.error_rate        0.0                      30                           0                         None                  0.0
15   cartservice-1   trace.from_frontend-2.error_rate        0.0                      30                           0                         None                  0.0
16   cartservice-2   trace.from_frontend-0.error_rate        0.0                      30                           0                         None                  0.0
17   cartservice-2   trace.from_frontend-1.error_rate        0.0                      30                           0                         None                  0.0
18   cartservice-2   trace.from_frontend-2.error_rate        0.0                      30                           0                         None                  0.0
19  cartservice2-0  trace.from_frontend2-0.error_rate        0.0                      30                           0                         None                  0.0,              cmdb_id  total_exceedances earliest_anomalous_timestamp_per_service
0        adservice-0                  0                                     None
1        adservice-1                  0                                     None
2        adservice-2                  0                                     None
3       adservice2-0                  0                                     None
4      cartservice-0                  0                                     None
5      cartservice-1                  0                                     None
6      cartservice-2                  0                                     None
7     cartservice2-0                  0                                     None
8  checkoutservice-0                  0                                     None
9  checkoutservice-1                  0                                     None)```
```